# Notebook 6 — Train, Tune, Evaluate

## Goal

Train and evaluate a machine learning model for late-delivery prediction.

In this notebook, we:
- start with a simple baseline,
- train a Logistic Regression model as a reference,
- train and tune an XGBoost model using the validation split,
- use metrics suitable for the imbalanced target,
- select a classification threshold using the validation set,
- evaluate the final selected model on the test set only once,
- save the trained model and the final results.

In [1]:
import os
import json
import joblib

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

## 1. Load the Notebook 5 Artifacts

Notebook 5 created the final processed feature tables and target files.

The preprocessing has already been fitted in Notebook 5, so no additional preprocessing is performed here.

In [2]:
artifact_dir = (
    "../artifacts/random_split/notebook5"
)

X_train = pd.read_csv(
    f"{artifact_dir}/train_features1.csv"
)

X_validation = pd.read_csv(
    f"{artifact_dir}/validation_features1.csv"
)

X_test = pd.read_csv(
    f"{artifact_dir}/test_features1.csv"
)

y_train = pd.read_csv(
    f"{artifact_dir}/train_target1.csv"
).iloc[:, 0]

y_validation = pd.read_csv(
    f"{artifact_dir}/validation_target1.csv"
).iloc[:, 0]

y_test = pd.read_csv(
    f"{artifact_dir}/test_target1.csv"
).iloc[:, 0]

with open(
    f"{artifact_dir}/feature_list1.json",
    "r",
    encoding="utf-8"
) as file:
    feature_names = json.load(file)

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


## 2. Check the Input Data

Before training, we verify that the feature matrices and target vectors are consistent.

In [3]:
print("Shapes")
print("=" * 50)

print("X_train     :", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test      :", X_test.shape)

print("\nTarget shapes")
print("=" * 50)

print("y_train     :", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test      :", y_test.shape)

print("\nFeature information")
print("=" * 50)

print("Feature list:", len(feature_names))
print("Matrix features:", X_train.shape[1])

Shapes
X_train     : (67529, 54)
X_validation: (14470, 54)
X_test      : (14471, 54)

Target shapes
y_train     : (67529,)
y_validation: (14470,)
y_test      : (14471,)

Feature information
Feature list: 54
Matrix features: 54


## 3. Validate the Dataset

The target is highly imbalanced, so we check its distribution before training.

In [4]:
def target_summary(y, name):
    counts = y.value_counts().sort_index()
    percentages = (
        y.value_counts(normalize=True)
        .sort_index()
        * 100
    )

    summary = pd.DataFrame({
        "count": counts,
        "percentage": percentages
    })

    summary.index.name = "late_delivery"

    print(f"\n{name}")
    print("=" * 50)

    display(
        summary.round(2)
    )


target_summary(
    y_train,
    "TRAIN"
)

target_summary(
    y_validation,
    "VALIDATION"
)

target_summary(
    y_test,
    "TEST"
)


TRAIN


,count,percentage
late_delivery,,
0,62051,91.89
1,5478,8.11



VALIDATION


,count,percentage
late_delivery,,
0,13296,91.89
1,1174,8.11



TEST


,count,percentage
late_delivery,,
0,13297,91.89
1,1174,8.11


## 4. Calculate the Class Imbalance Weight

The late-delivery class is much smaller than the on-time class.

For XGBoost, `scale_pos_weight` can give more importance to the minority class during training.

In [5]:
negative_count = (
    y_train == 0
).sum()

positive_count = (
    y_train == 1
).sum()

scale_pos_weight = (
    negative_count /
    positive_count
)

print(
    "Negative samples:",
    negative_count
)

print(
    "Positive samples:",
    positive_count
)

print(
    "scale_pos_weight:",
    round(
        scale_pos_weight,
        4
    )
)

Negative samples: 62051
Positive samples: 5478
scale_pos_weight: 11.3273


## 5. Check the Input Data for Errors

The final model inputs should not contain missing or infinite values.

In [6]:
print("Missing values")
print("=" * 50)

print(
    "Train:",
    X_train.isna().sum().sum()
)

print(
    "Validation:",
    X_validation.isna().sum().sum()
)

print(
    "Test:",
    X_test.isna().sum().sum()
)

print("\nInfinite values")
print("=" * 50)

print(
    "Train:",
    np.isinf(X_train.to_numpy()).sum()
)

print(
    "Validation:",
    np.isinf(X_validation.to_numpy()).sum()
)

print(
    "Test:",
    np.isinf(X_test.to_numpy()).sum()
)

Missing values
Train: 0
Validation: 0
Test: 0

Infinite values
Train: 0
Validation: 0
Test: 0


## 6. Define Evaluation Metrics

Accuracy alone is not suitable for this problem because the late-delivery class is relatively rare.

PR-AUC is used as the main model-selection metric because it focuses on the positive class.
Precision, recall, and F1 are also reported to understand the classification trade-off.

In [7]:
def evaluate_predictions(
    y_true,
    probabilities,
    threshold=0.50
):

    predictions = (
        probabilities >= threshold
    ).astype(int)

    results = {
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probabilities
        ),
        "PR_AUC": average_precision_score(
            y_true,
            probabilities
        )
    }

    return results, predictions

## 7. Train a Simple Baseline

The baseline always predicts the majority class.

This gives us a minimum reference point that every useful model should beat.

In [8]:
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(
    X_train,
    y_train
)

baseline_prob = (
    baseline.predict_proba(
        X_validation
    )[:, 1]
)

baseline_results, baseline_pred = (
    evaluate_predictions(
        y_validation,
        baseline_prob,
        threshold=0.50
    )
)

print("Baseline Performance")
print("=" * 60)

for metric, value in baseline_results.items():
    print(
        f"{metric}:",
        round(value, 4)
        if isinstance(value, (float, np.floating))
        else value
    )

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        baseline_pred
    )
)

Baseline Performance
Threshold: 0.5
Accuracy: 0.9189
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC_AUC: 0.5
PR_AUC: 0.0811

Confusion Matrix:
[[13296     0]
 [ 1174     0]]


## 8. Train Logistic Regression

Logistic Regression provides a simple machine learning reference before moving to the more flexible XGBoost model.

Class weighting is used because of the class imbalance.

In [9]:
logreg = LogisticRegression(
    solver="liblinear",
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

logreg.fit(
    X_train,
    y_train
)

logreg_prob = (
    logreg.predict_proba(
        X_validation
    )[:, 1]
)

logreg_results, logreg_pred = (
    evaluate_predictions(
        y_validation,
        logreg_prob,
        threshold=0.50
    )
)

print("Logistic Regression")
print("=" * 60)

for metric, value in logreg_results.items():
    print(
        f"{metric}:",
        round(value, 4)
        if isinstance(value, (float, np.floating))
        else value
    )

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        logreg_pred
    )
)

Logistic Regression
Threshold: 0.5
Accuracy: 0.656
Precision: 0.1415
Recall: 0.6397
F1: 0.2318
ROC_AUC: 0.6986
PR_AUC: 0.1826

Confusion Matrix:
[[8741 4555]
 [ 423  751]]


## 9. Compare the Baseline and Logistic Regression

The comparison shows whether the first machine learning model provides useful improvement over the simple baseline.

In [10]:
baseline_logreg_comparison = pd.DataFrame([
    {
        "Model": "Baseline",
        **baseline_results
    },
    {
        "Model": "Logistic Regression",
        **logreg_results
    }
])

display(
    baseline_logreg_comparison.round(4)
)

,Model,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Baseline,0.5,0.9189,0.0000,0.0000,0.0000,0.5000,0.0811
1,Logistic Regression,0.5,0.6560,0.1415,0.6397,0.2318,0.6986,0.1826


## 10. Tune XGBoost on the Validation Set

We now try several XGBoost configurations.

The configurations differ in tree depth, learning rate, number of trees, minimum child weight, subsampling, and regularization.

Each model is trained on the training split and evaluated on the validation split.
PR-AUC is used to select the best configuration.

In [11]:
xgb_configs = [

    {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_child_weight": 5,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0
    },

    {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_child_weight": 5,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0
    },

    {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0
    },

    {
        "n_estimators": 700,
        "learning_rate": 0.03,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0
    },

    {
        "n_estimators": 500,
        "learning_rate": 0.02,
        "max_depth": 4,
        "min_child_weight": 8,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0,
        "reg_alpha": 0.0
    },

    {
        "n_estimators": 700,
        "learning_rate": 0.02,
        "max_depth": 5,
        "min_child_weight": 8,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0,
        "reg_alpha": 0.0
    },

    {
        "n_estimators": 700,
        "learning_rate": 0.03,
        "max_depth": 5,
        "min_child_weight": 8,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 15.0,
        "reg_alpha": 0.0
    },

    {
        "n_estimators": 800,
        "learning_rate": 0.02,
        "max_depth": 4,
        "min_child_weight": 10,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 20.0,
        "reg_alpha": 0.5
    }
]

print(
    "Number of XGBoost configurations:",
    len(xgb_configs)
)

Number of XGBoost configurations: 8


## 11. Run the XGBoost Configurations

Each configuration is evaluated on the validation set.

The test set is not used during this process.

In [12]:
tuning_results = []
tuned_models = []

for i, config in enumerate(
    xgb_configs,
    start=1
):

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        min_child_weight=config["min_child_weight"],

        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],

        reg_lambda=config["reg_lambda"],
        reg_alpha=config["reg_alpha"],

        scale_pos_weight=scale_pos_weight,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_validation,
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = (
        model.predict_proba(
            X_validation
        )[:, 1]
    )

    pr_auc = (
        average_precision_score(
            y_validation,
            probabilities
        )
    )

    roc_auc = (
        roc_auc_score(
            y_validation,
            probabilities
        )
    )

    tuned_models.append(model)

    tuning_results.append({
        "configuration": i,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        **config
    })

    print(
        f"Configuration {i}: "
        f"PR-AUC={pr_auc:.4f}, "
        f"ROC-AUC={roc_auc:.4f}"
    )


xgb_tuning_results = (
    pd.DataFrame(tuning_results)
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nXGBoost tuning results:")
display(
    xgb_tuning_results.round(4)
)

Configuration 1: PR-AUC=0.2901, ROC-AUC=0.7793
Configuration 2: PR-AUC=0.2961, ROC-AUC=0.7822
Configuration 3: PR-AUC=0.3097, ROC-AUC=0.7897
Configuration 4: PR-AUC=0.3151, ROC-AUC=0.7915
Configuration 5: PR-AUC=0.3083, ROC-AUC=0.7878
Configuration 6: PR-AUC=0.3257, ROC-AUC=0.7958
Configuration 7: PR-AUC=0.3247, ROC-AUC=0.7963
Configuration 8: PR-AUC=0.3123, ROC-AUC=0.7904

XGBoost tuning results:


,configuration,pr_auc,roc_auc,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha
0,6,0.3257,0.7958,700,0.02,5,8,0.90,0.90,15.0,0.0
1,7,0.3247,0.7963,700,0.03,5,8,0.85,0.85,15.0,0.0
2,4,0.3151,0.7915,700,0.03,4,5,0.90,0.90,10.0,0.0
3,8,0.3123,0.7904,800,0.02,4,10,0.85,0.85,20.0,0.5
4,3,0.3097,0.7897,500,0.03,4,5,0.90,0.90,10.0,0.0
5,5,0.3083,0.7878,500,0.02,4,8,0.90,0.90,15.0,0.0
6,2,0.2961,0.7822,500,0.03,3,5,0.90,0.90,10.0,0.0
7,1,0.2901,0.7793,300,0.03,3,5,0.90,0.90,10.0,0.0


## 12. Select the Best XGBoost Configuration

The configuration with the highest validation PR-AUC is selected for further evaluation.

In [14]:
best_configuration_id = int(
    xgb_tuning_results.loc[
        0,
        "configuration"
    ]
)

best_xgb_model = tuned_models[
    best_configuration_id - 1
]

best_xgb_row = (
    xgb_tuning_results.iloc[0]
)

print(
    "Best configuration:",
    best_configuration_id
)

print(
    "Validation PR-AUC:",
    round(
        best_xgb_row["pr_auc"],
        4
    )
)

print(
    "Validation ROC-AUC:",
    round(
        best_xgb_row["roc_auc"],
        4
    )
)

Best configuration: 6
Validation PR-AUC: 0.3257
Validation ROC-AUC: 0.7958


## 13. Evaluate the Best XGBoost Model at Threshold 0.50

The default threshold is checked before selecting a better threshold.

In [15]:
xgb_prob = (
    best_xgb_model
    .predict_proba(
        X_validation
    )[:, 1]
)

xgb_results_05, xgb_pred_05 = (
    evaluate_predictions(
        y_validation,
        xgb_prob,
        threshold=0.50
    )
)

print("Best XGBoost — Threshold 0.50")
print("=" * 60)

for metric, value in xgb_results_05.items():
    print(
        f"{metric}:",
        round(value, 4)
        if isinstance(value, (float, np.floating))
        else value
    )

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        xgb_pred_05
    )
)

Best XGBoost — Threshold 0.50
Threshold: 0.5
Accuracy: 0.772
Precision: 0.2128
Recall: 0.6704
F1: 0.323
ROC_AUC: 0.7958
PR_AUC: 0.3257

Confusion Matrix:
[[10384  2912]
 [  387   787]]


## 14. Tune the Classification Threshold

The model produces probabilities, but the final prediction depends on the classification threshold.

The threshold is selected using the validation set.

F1 is used here because it balances precision and recall for the minority class.

In [16]:
thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

threshold_results = []

for threshold in thresholds:

    predictions = (
        xgb_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,

        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        )
    })

xgb_threshold_df = pd.DataFrame(
    threshold_results
)

best_threshold_index = (
    xgb_threshold_df["f1"]
    .idxmax()
)

best_threshold = (
    xgb_threshold_df
    .loc[
        best_threshold_index,
        "threshold"
    ]
)

print(
    "Best threshold:",
    round(best_threshold, 2)
)

print(
    "Best validation F1:",
    round(
        xgb_threshold_df
        .loc[
            best_threshold_index,
            "f1"
        ],
        4
    )
)

Best threshold: 0.69
Best validation F1: 0.3806


## 15. Review the Threshold Trade-off

We inspect several threshold values to understand how precision and recall change.

In [17]:
selected_thresholds = (
    xgb_threshold_df[
        xgb_threshold_df["threshold"].isin(
            [
                0.10,
                0.20,
                0.30,
                0.40,
                0.50,
                0.60,
                0.70,
                0.80
            ]
        )
    ]
)

display(
    selected_thresholds.round(4)
)

,threshold,precision,recall,f1
5,0.1,0.0863,0.9889,0.1588
15,0.2,0.1013,0.9506,0.1831


## 16. Evaluate XGBoost at the Selected Threshold

The selected threshold is now applied to the validation predictions.

In [18]:
xgb_best_pred = (
    xgb_prob >= best_threshold
).astype(int)

xgb_best_results = {
    "Threshold": best_threshold,

    "Accuracy": accuracy_score(
        y_validation,
        xgb_best_pred
    ),

    "Precision": precision_score(
        y_validation,
        xgb_best_pred,
        zero_division=0
    ),

    "Recall": recall_score(
        y_validation,
        xgb_best_pred,
        zero_division=0
    ),

    "F1": f1_score(
        y_validation,
        xgb_best_pred,
        zero_division=0
    ),

    "ROC_AUC": roc_auc_score(
        y_validation,
        xgb_prob
    ),

    "PR_AUC": average_precision_score(
        y_validation,
        xgb_prob
    )
}

print("Best XGBoost — Selected Threshold")
print("=" * 60)

for metric, value in xgb_best_results.items():
    print(
        f"{metric}:",
        round(value, 4)
        if isinstance(value, (float, np.floating))
        else value
    )

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        xgb_best_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        xgb_best_pred,
        target_names=[
            "On-time",
            "Late"
        ],
        zero_division=0
    )
)

Best XGBoost — Selected Threshold
Threshold: 0.69
Accuracy: 0.8826
Precision: 0.3327
Recall: 0.4446
F1: 0.3806
ROC_AUC: 0.7958
PR_AUC: 0.3257

Confusion Matrix:
[[12249  1047]
 [  652   522]]

Classification Report:
              precision    recall  f1-score   support

     On-time       0.95      0.92      0.94     13296
        Late       0.33      0.44      0.38      1174

    accuracy                           0.88     14470
   macro avg       0.64      0.68      0.66     14470
weighted avg       0.90      0.88      0.89     14470



## 17. Compare the Main Models

The comparison includes the baseline, Logistic Regression, and the tuned XGBoost model.

PR-AUC is the main metric for comparing the models because the target is imbalanced.

In [19]:
model_comparison = pd.DataFrame([

    {
        "Model": "Baseline",
        "Threshold": 0.50,
        "Precision": baseline_results["Precision"],
        "Recall": baseline_results["Recall"],
        "F1": baseline_results["F1"],
        "ROC_AUC": baseline_results["ROC_AUC"],
        "PR_AUC": baseline_results["PR_AUC"]
    },

    {
        "Model": "Logistic Regression",
        "Threshold": 0.50,
        "Precision": logreg_results["Precision"],
        "Recall": logreg_results["Recall"],
        "F1": logreg_results["F1"],
        "ROC_AUC": logreg_results["ROC_AUC"],
        "PR_AUC": logreg_results["PR_AUC"]
    },

    {
        "Model": "Tuned XGBoost",
        "Threshold": best_threshold,
        "Precision": xgb_best_results["Precision"],
        "Recall": xgb_best_results["Recall"],
        "F1": xgb_best_results["F1"],
        "ROC_AUC": xgb_best_results["ROC_AUC"],
        "PR_AUC": xgb_best_results["PR_AUC"]
    }
])

display(
    model_comparison.round(4)
)

,Model,Threshold,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Baseline,0.50,0.0000,0.0000,0.0000,0.5000,0.0811
1,Logistic Regression,0.50,0.1415,0.6397,0.2318,0.6986,0.1826
2,Tuned XGBoost,0.69,0.3327,0.4446,0.3806,0.7958,0.3257


## 18. Select a Strong XGBoost Candidate

The initial XGBoost tuning identifies a strong candidate model based on validation performance.

This model is not yet considered the final model because additional feature selection, class-weight tuning, threshold optimization, and model-combination experiments are performed in the following stages.

The validation set is used throughout the tuning process, while the test set remains untouched.

In [20]:
best_model_name = "Strong XGBoost Candidate"
best_model = best_xgb_model
selected_threshold = best_threshold

best_validation_pr_auc = (
    xgb_best_results["PR_AUC"]
)

best_validation_f1 = (
    xgb_best_results["F1"]
)

print(
    "Selected model:",
    best_model_name
)

print(
    "Selected threshold:",
    round(
        selected_threshold,
        4
    )
)

print(
    "Validation PR-AUC:",
    round(
        best_validation_pr_auc,
        4
    )
)

print(
    "Validation F1:",
    round(
        best_validation_f1,
        4
    )
)

Selected model: Tuned XGBoost
Selected threshold: 0.69
Validation PR-AUC: 0.3257
Validation F1: 0.3806


## 19. Tune XGBoost Class Weight

The default class imbalance ratio is not necessarily the optimal value for maximizing the F1-score of the minority class.

Several `scale_pos_weight` values are tested using the same feature representation and model structure. Each configuration is evaluated on the validation set, and the classification threshold is optimized separately.

The test set remains completely untouched during this experiment.


In [22]:
print("=" * 80)
print("XGBOOST CLASS-WEIGHT TUNING")
print("=" * 80)

weight_values = [
    1.0,
    2.0,
    3.0,
    5.0,
    7.0,
    10.0,
    12.0,
    15.0
]

weight_results = []
weight_models = {}

for weight in weight_values:

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=700,
        learning_rate=0.02,
        max_depth=5,
        min_child_weight=8,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=15.0,
        reg_alpha=0.0,

        scale_pos_weight=weight,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_validation,
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = (
        model
        .predict_proba(X_validation)[:, 1]
    )

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    # Find the best F1 threshold for this weight
    best_f1 = 0
    best_threshold_for_weight = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.01
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold_for_weight = threshold
            best_precision = precision
            best_recall = recall

    weight_models[weight] = model

    weight_results.append({
        "scale_pos_weight": weight,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best_Threshold": best_threshold_for_weight,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })

    print(
        f"Weight {weight:>5}: "
        f"PR-AUC={pr_auc:.4f} | "
        f"Best F1={best_f1:.4f} | "
        f"Threshold={best_threshold_for_weight:.2f}"
    )


weight_results_df = (
    pd.DataFrame(weight_results)
    .sort_values(
        ["F1", "PR_AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nResults:")
display(
    weight_results_df.round(4)
)

print("\nBest configuration:")
display(
    weight_results_df.head(1).round(4)
)

XGBOOST CLASS-WEIGHT TUNING
Weight   1.0: PR-AUC=0.3268 | Best F1=0.3783 | Threshold=0.16
Weight   2.0: PR-AUC=0.3285 | Best F1=0.3800 | Threshold=0.25
Weight   3.0: PR-AUC=0.3270 | Best F1=0.3788 | Threshold=0.38
Weight   5.0: PR-AUC=0.3274 | Best F1=0.3824 | Threshold=0.50
Weight   7.0: PR-AUC=0.3271 | Best F1=0.3811 | Threshold=0.57
Weight  10.0: PR-AUC=0.3246 | Best F1=0.3806 | Threshold=0.66
Weight  12.0: PR-AUC=0.3234 | Best F1=0.3812 | Threshold=0.70
Weight  15.0: PR-AUC=0.3240 | Best F1=0.3808 | Threshold=0.74

Results:


,scale_pos_weight,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,5.0,0.3274,0.7957,0.50,0.3294,0.4557,0.3824
1,12.0,0.3234,0.7958,0.70,0.3327,0.4463,0.3812
2,7.0,0.3271,0.7954,0.57,0.3220,0.4668,0.3811
3,15.0,0.3240,0.7953,0.74,0.3302,0.4497,0.3808
4,10.0,0.3246,0.7953,0.66,0.3304,0.4489,0.3806
5,2.0,0.3285,0.7946,0.25,0.2951,0.5332,0.3800
6,3.0,0.3270,0.7946,0.38,0.3264,0.4514,0.3788
7,1.0,0.3268,0.7933,0.16,0.3089,0.4881,0.3783



Best configuration:


,scale_pos_weight,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,5.0,0.3274,0.7957,0.5,0.3294,0.4557,0.3824


## 20. Feature Group Ablation

Different groups of features are evaluated to determine which types of information contribute most to late-delivery prediction.

The experiment compares the full feature set with reduced sets containing delivery-related, financial, product-related, and geographic/time features.

This helps identify whether removing certain groups can improve generalization and reduce unnecessary model complexity.

All comparisons are performed using the training and validation sets only.

In [23]:
print("=" * 80)
print("FEATURE GROUP ABLATION TEST")
print("=" * 80)

feature_groups = {
    "All features": X_train.columns.tolist(),

    "Core delivery features": [
        col for col in X_train.columns
        if any(
            key in col.lower()
            for key in [
                "distance",
                "same_state",
                "estimated_delivery",
                "purchase_month",
                "purchase_year",
                "purchase_dayofweek",
                "purchase_hour",
                "purchase_dayofmonth",
                "is_weekend",
                "is_holiday",
                "number_of_sellers"
            ]
        )
    ],

    "Without financial features": [
        col for col in X_train.columns
        if not any(
            key in col.lower()
            for key in [
                "total_price",
                "total_freight",
                "total_payment",
                "installments",
                "number_of_payments"
            ]
        )
    ],

    "Without product features": [
        col for col in X_train.columns
        if not any(
            key in col.lower()
            for key in [
                "product_weight",
                "product_volume",
                "number_of_product_categories",
                "number_of_items"
            ]
        )
    ],

    "Geography + time only": [
        col for col in X_train.columns
        if any(
            key in col.lower()
            for key in [
                "customer_state",
                "customer_city",
                "customer_zip",
                "distance",
                "same_state",
                "purchase_year",
                "purchase_month",
                "purchase_dayofweek",
                "purchase_hour",
                "purchase_dayofmonth",
                "weekend",
                "holiday",
                "estimated_delivery"
            ]
        )
    ]
}

ablation_results = []

for group_name, features in feature_groups.items():

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=700,
        learning_rate=0.02,
        max_depth=5,
        min_child_weight=8,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=15.0,
        reg_alpha=0.0,

        scale_pos_weight=5.0,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train[features],
        y_train,
        eval_set=[
            (
                X_validation[features],
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = model.predict_proba(
        X_validation[features]
    )[:, 1]

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.01
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    ablation_results.append({
        "Feature Group": group_name,
        "Feature Count": len(features),
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })

ablation_df = (
    pd.DataFrame(ablation_results)
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    ablation_df.round(4)
)

FEATURE GROUP ABLATION TEST


,Feature Group,Feature Count,PR_AUC,ROC_AUC,Best Threshold,Precision,Recall,F1
0,Without product features,50,0.3273,0.7972,0.53,0.3499,0.4267,0.3845
1,All features,54,0.3274,0.7957,0.50,0.3294,0.4557,0.3824
2,Geography + time only,39,0.3316,0.7977,0.47,0.3074,0.4983,0.3802
3,Without financial features,48,0.3291,0.7961,0.50,0.3280,0.4523,0.3802
4,Core delivery features,11,0.2818,0.7781,0.52,0.2994,0.4114,0.3466


## 21. Feature Selection

Feature importance from an XGBoost model is used to rank the available features.

Several feature subset sizes are then tested to determine whether a smaller set of informative features can perform as well as or better than the full feature set.

The purpose is to reduce unnecessary features while maintaining or improving validation performance.

In [24]:
print("=" * 80)
print("XGBOOST FEATURE SELECTION TEST")
print("=" * 80)

# Use the current best configuration
base_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",

    n_estimators=700,
    learning_rate=0.02,
    max_depth=5,
    min_child_weight=8,

    subsample=0.90,
    colsample_bytree=0.90,

    reg_lambda=15.0,
    reg_alpha=0.0,

    scale_pos_weight=5.0,

    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

base_model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_validation, y_validation)
    ],
    verbose=False
)

# Rank features by importance
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": base_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("\nTop 20 features:")
display(
    importance_df.head(20).round(6)
)


# Test different feature counts
feature_counts = [
    10,
    15,
    20,
    25,
    30,
    35,
    40,
    45,
    50,
    54
]

feature_selection_results = []
feature_selection_models = {}

for count in feature_counts:

    selected_features = (
        importance_df
        .head(count)["feature"]
        .tolist()
    )

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=700,
        learning_rate=0.02,
        max_depth=5,
        min_child_weight=8,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=15.0,
        reg_alpha=0.0,

        scale_pos_weight=5.0,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train[selected_features],
        y_train,
        eval_set=[
            (
                X_validation[selected_features],
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = (
        model
        .predict_proba(
            X_validation[selected_features]
        )[:, 1]
    )

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.01
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    feature_selection_models[count] = model

    feature_selection_results.append({
        "Feature_Count": count,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best_Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })

feature_selection_df = (
    pd.DataFrame(
        feature_selection_results
    )
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFeature selection results:")
display(
    feature_selection_df.round(4)
)

XGBOOST FEATURE SELECTION TEST

Top 20 features:


,feature,importance
0,onehot_customer_state_SP,0.093075
1,onehot_customer_state_RJ,0.084800
2,num_purchase_year,0.059340
3,num_purchase_month,0.058259
4,num_customer_seller_same_state,0.054139
5,num_estimated_delivery_days,0.044305
6,onehot_customer_state_MG,0.044132
7,onehot_customer_state_PR,0.043559
8,num_customer_seller_distance_km,0.028365
9,num_number_of_sellers,0.027309



Feature selection results:


,Feature_Count,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,30,0.3282,0.7965,0.47,0.3136,0.5000,0.3854
1,54,0.3295,0.7959,0.51,0.3368,0.4463,0.3839
2,35,0.3294,0.7965,0.46,0.3050,0.5111,0.3820
3,20,0.3259,0.7968,0.51,0.3284,0.4549,0.3814
4,40,0.3311,0.7967,0.47,0.3078,0.4991,0.3808
5,45,0.3281,0.7957,0.45,0.2976,0.5256,0.3800
6,50,0.3289,0.7970,0.49,0.3208,0.4659,0.3800
7,25,0.3229,0.7935,0.54,0.3480,0.4106,0.3767
8,15,0.2964,0.7870,0.52,0.3136,0.4336,0.3640
9,10,0.2957,0.7860,0.54,0.3245,0.4080,0.3615


## 22. Focused XGBoost Tuning

After evaluating different feature subsets, a focused search is performed around the strongest feature groups.

The experiment combines different feature subset sizes, class weights, and tree depths to identify a stronger XGBoost configuration.

Model selection is based on validation performance, with particular attention to F1-score and PR-AUC.

In [25]:
print("=" * 80)
print("FOCUSED XGBOOST TUNING")
print("=" * 80)

# Start from the feature ranking obtained from the current model
ranked_features = (
    importance_df["feature"].tolist()
)

feature_sets = {
    "Top 30": ranked_features[:30],
    "Top 40": ranked_features[:40],
    "All 54": ranked_features[:54]
}

weights = [
    4.0,
    5.0,
    6.0
]

depths = [
    4,
    5
]

tuning_results = []
tuning_models = {}

configuration_id = 0

for feature_set_name, selected_features in feature_sets.items():

    for weight in weights:

        for depth in depths:

            configuration_id += 1

            model = xgb.XGBClassifier(
                objective="binary:logistic",
                eval_metric="aucpr",

                n_estimators=900,
                learning_rate=0.02,

                max_depth=depth,
                min_child_weight=8,

                subsample=0.90,
                colsample_bytree=0.90,

                reg_lambda=15.0,
                reg_alpha=0.0,

                scale_pos_weight=weight,

                random_state=42,
                n_jobs=-1,
                tree_method="hist"
            )

            model.fit(
                X_train[selected_features],
                y_train,
                eval_set=[
                    (
                        X_validation[selected_features],
                        y_validation
                    )
                ],
                verbose=False
            )

            probabilities = (
                model
                .predict_proba(
                    X_validation[selected_features]
                )[:, 1]
            )

            pr_auc = (
                average_precision_score(
                    y_validation,
                    probabilities
                )
            )

            roc_auc = (
                roc_auc_score(
                    y_validation,
                    probabilities
                )
            )

            best_f1 = 0
            best_threshold = 0
            best_precision = 0
            best_recall = 0

            for threshold in np.arange(
                0.05,
                0.96,
                0.01
            ):

                predictions = (
                    probabilities >= threshold
                ).astype(int)

                precision = precision_score(
                    y_validation,
                    predictions,
                    zero_division=0
                )

                recall = recall_score(
                    y_validation,
                    predictions,
                    zero_division=0
                )

                f1 = f1_score(
                    y_validation,
                    predictions,
                    zero_division=0
                )

                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = threshold
                    best_precision = precision
                    best_recall = recall

            tuning_models[configuration_id] = {
                "model": model,
                "features": selected_features
            }

            tuning_results.append({
                "configuration": configuration_id,
                "feature_set": feature_set_name,
                "feature_count": len(selected_features),
                "scale_pos_weight": weight,
                "max_depth": depth,
                "n_estimators": 900,
                "learning_rate": 0.02,
                "PR_AUC": pr_auc,
                "ROC_AUC": roc_auc,
                "Best_Threshold": best_threshold,
                "Precision": best_precision,
                "Recall": best_recall,
                "F1": best_f1
            })

            print(
                f"Config {configuration_id:02d} | "
                f"{feature_set_name:7s} | "
                f"weight={weight:.1f} | "
                f"depth={depth} | "
                f"PR-AUC={pr_auc:.4f} | "
                f"F1={best_f1:.4f}"
            )


focused_tuning_df = (
    pd.DataFrame(tuning_results)
    .sort_values(
        ["F1", "PR_AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTop configurations:")
display(
    focused_tuning_df.head(12).round(4)
)

print("\nBest configuration:")
display(
    focused_tuning_df.head(1).round(4)
)

FOCUSED XGBOOST TUNING
Config 01 | Top 30  | weight=4.0 | depth=4 | PR-AUC=0.3167 | F1=0.3745
Config 02 | Top 30  | weight=4.0 | depth=5 | PR-AUC=0.3315 | F1=0.3838
Config 03 | Top 30  | weight=5.0 | depth=4 | PR-AUC=0.3173 | F1=0.3769
Config 04 | Top 30  | weight=5.0 | depth=5 | PR-AUC=0.3312 | F1=0.3835
Config 05 | Top 30  | weight=6.0 | depth=4 | PR-AUC=0.3159 | F1=0.3735
Config 06 | Top 30  | weight=6.0 | depth=5 | PR-AUC=0.3323 | F1=0.3853
Config 07 | Top 40  | weight=4.0 | depth=4 | PR-AUC=0.3184 | F1=0.3737
Config 08 | Top 40  | weight=4.0 | depth=5 | PR-AUC=0.3336 | F1=0.3849
Config 09 | Top 40  | weight=5.0 | depth=4 | PR-AUC=0.3180 | F1=0.3725
Config 10 | Top 40  | weight=5.0 | depth=5 | PR-AUC=0.3336 | F1=0.3849
Config 11 | Top 40  | weight=6.0 | depth=4 | PR-AUC=0.3188 | F1=0.3724
Config 12 | Top 40  | weight=6.0 | depth=5 | PR-AUC=0.3347 | F1=0.3842
Config 13 | All 54  | weight=4.0 | depth=4 | PR-AUC=0.3169 | F1=0.3750
Config 14 | All 54  | weight=4.0 | depth=5 | PR-AUC=0.

,configuration,feature_set,feature_count,scale_pos_weight,max_depth,n_estimators,learning_rate,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,6,Top 30,30,6.0,5,900,0.02,0.3323,0.7978,0.49,0.3022,0.5315,0.3853
1,10,Top 40,40,5.0,5,900,0.02,0.3336,0.7981,0.50,0.3301,0.4617,0.3849
2,8,Top 40,40,4.0,5,900,0.02,0.3336,0.7984,0.46,0.3374,0.4480,0.3849
3,12,Top 40,40,6.0,5,900,0.02,0.3347,0.7986,0.56,0.3422,0.4378,0.3842
4,16,All 54,54,5.0,5,900,0.02,0.3309,0.7974,0.53,0.3515,0.4233,0.3841
5,2,Top 30,30,4.0,5,900,0.02,0.3315,0.7974,0.44,0.3227,0.4736,0.3838
6,4,Top 30,30,5.0,5,900,0.02,0.3312,0.7981,0.49,0.3214,0.4753,0.3835
7,18,All 54,54,6.0,5,900,0.02,0.3310,0.7978,0.56,0.3421,0.4336,0.3824
8,14,All 54,54,4.0,5,900,0.02,0.3293,0.7969,0.47,0.3431,0.4310,0.3820
9,3,Top 30,30,5.0,4,900,0.02,0.3173,0.7920,0.48,0.3098,0.4813,0.3769



Best configuration:


,configuration,feature_set,feature_count,scale_pos_weight,max_depth,n_estimators,learning_rate,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,6,Top 30,30,6.0,5,900,0.02,0.3323,0.7978,0.49,0.3022,0.5315,0.3853


## 23. Tree Structure Tuning

The structure of the XGBoost trees is further investigated by varying `max_depth` and `min_child_weight`.

These parameters control the complexity of the decision trees and can affect the balance between model flexibility and overfitting.

The experiment is conducted on the strongest feature subsets identified previously.

In [26]:
print("=" * 80)
print("XGBOOST TREE STRUCTURE TUNING")
print("=" * 80)

# Use the two strongest feature sets found so far
feature_sets = {
    "Top 30": ranked_features[:30],
    "Top 40": ranked_features[:40]
}

depth_values = [
    5,
    6,
    7
]

min_child_values = [
    3,
    8,
    12
]

tuning_results = []
tuning_models = {}

configuration_id = 0

for feature_set_name, selected_features in feature_sets.items():

    for depth in depth_values:

        for min_child in min_child_values:

            configuration_id += 1

            model = xgb.XGBClassifier(
                objective="binary:logistic",
                eval_metric="aucpr",

                n_estimators=900,
                learning_rate=0.02,

                max_depth=depth,
                min_child_weight=min_child,

                gamma=0.0,

                subsample=0.90,
                colsample_bytree=0.90,

                reg_lambda=15.0,
                reg_alpha=0.0,

                scale_pos_weight=6.0,

                random_state=42,
                n_jobs=-1,
                tree_method="hist"
            )

            model.fit(
                X_train[selected_features],
                y_train,
                eval_set=[
                    (
                        X_validation[selected_features],
                        y_validation
                    )
                ],
                verbose=False
            )

            probabilities = (
                model
                .predict_proba(
                    X_validation[selected_features]
                )[:, 1]
            )

            pr_auc = average_precision_score(
                y_validation,
                probabilities
            )

            roc_auc = roc_auc_score(
                y_validation,
                probabilities
            )

            best_f1 = 0
            best_threshold = 0
            best_precision = 0
            best_recall = 0

            for threshold in np.arange(
                0.05,
                0.96,
                0.01
            ):

                predictions = (
                    probabilities >= threshold
                ).astype(int)

                precision = precision_score(
                    y_validation,
                    predictions,
                    zero_division=0
                )

                recall = recall_score(
                    y_validation,
                    predictions,
                    zero_division=0
                )

                f1 = f1_score(
                    y_validation,
                    predictions,
                    zero_division=0
                )

                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = threshold
                    best_precision = precision
                    best_recall = recall

            tuning_models[configuration_id] = {
                "model": model,
                "features": selected_features
            }

            tuning_results.append({
                "configuration": configuration_id,
                "feature_set": feature_set_name,
                "feature_count": len(selected_features),
                "max_depth": depth,
                "min_child_weight": min_child,
                "scale_pos_weight": 6.0,
                "PR_AUC": pr_auc,
                "ROC_AUC": roc_auc,
                "Best_Threshold": best_threshold,
                "Precision": best_precision,
                "Recall": best_recall,
                "F1": best_f1
            })

            print(
                f"Config {configuration_id:02d} | "
                f"{feature_set_name:7s} | "
                f"depth={depth} | "
                f"min_child={min_child} | "
                f"PR-AUC={pr_auc:.4f} | "
                f"F1={best_f1:.4f}"
            )


tree_tuning_df = (
    pd.DataFrame(tuning_results)
    .sort_values(
        ["F1", "PR_AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTop configurations:")
display(
    tree_tuning_df.head(12).round(4)
)

print("\nBest configuration:")
display(
    tree_tuning_df.head(1).round(4)
)

XGBOOST TREE STRUCTURE TUNING
Config 01 | Top 30  | depth=5 | min_child=3 | PR-AUC=0.3317 | F1=0.3833
Config 02 | Top 30  | depth=5 | min_child=8 | PR-AUC=0.3323 | F1=0.3853
Config 03 | Top 30  | depth=5 | min_child=12 | PR-AUC=0.3325 | F1=0.3844
Config 04 | Top 30  | depth=6 | min_child=3 | PR-AUC=0.3371 | F1=0.3912
Config 05 | Top 30  | depth=6 | min_child=8 | PR-AUC=0.3365 | F1=0.3869
Config 06 | Top 30  | depth=6 | min_child=12 | PR-AUC=0.3359 | F1=0.3889
Config 07 | Top 30  | depth=7 | min_child=3 | PR-AUC=0.3399 | F1=0.3894
Config 08 | Top 30  | depth=7 | min_child=8 | PR-AUC=0.3388 | F1=0.3897
Config 09 | Top 30  | depth=7 | min_child=12 | PR-AUC=0.3384 | F1=0.3910
Config 10 | Top 40  | depth=5 | min_child=3 | PR-AUC=0.3321 | F1=0.3837
Config 11 | Top 40  | depth=5 | min_child=8 | PR-AUC=0.3347 | F1=0.3842
Config 12 | Top 40  | depth=5 | min_child=12 | PR-AUC=0.3324 | F1=0.3810
Config 13 | Top 40  | depth=6 | min_child=3 | PR-AUC=0.3391 | F1=0.3870
Config 14 | Top 40  | depth=6 

,configuration,feature_set,feature_count,max_depth,min_child_weight,scale_pos_weight,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,16,Top 40,40,7,3,6.0,0.3406,0.8026,0.53,0.3453,0.4591,0.3941
1,4,Top 30,30,6,3,6.0,0.3371,0.8011,0.52,0.3272,0.4864,0.3912
2,9,Top 30,30,7,12,6.0,0.3384,0.8029,0.52,0.3300,0.4796,0.3910
3,15,Top 40,40,6,12,6.0,0.3394,0.8017,0.54,0.3364,0.4634,0.3898
4,8,Top 30,30,7,8,6.0,0.3388,0.8025,0.52,0.3294,0.4770,0.3897
5,14,Top 40,40,6,8,6.0,0.3390,0.8015,0.53,0.3311,0.4727,0.3895
6,7,Top 30,30,7,3,6.0,0.3399,0.8017,0.56,0.3587,0.4259,0.3894
7,6,Top 30,30,6,12,6.0,0.3359,0.8003,0.54,0.3341,0.4651,0.3889
8,18,Top 40,40,7,12,6.0,0.3404,0.8026,0.51,0.3244,0.4830,0.3881
9,17,Top 40,40,7,8,6.0,0.3408,0.8024,0.50,0.3193,0.4923,0.3874



Best configuration:


,configuration,feature_set,feature_count,max_depth,min_child_weight,scale_pos_weight,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,16,Top 40,40,7,3,6.0,0.3406,0.8026,0.53,0.3453,0.4591,0.3941


## 24. Fine-Tune the Best XGBoost Configuration

The strongest configuration identified so far is fine-tuned by making smaller adjustments to the learning rate, number of estimators, subsampling, column sampling, tree depth, and regularization.

The purpose is to determine whether small parameter changes can provide additional validation improvement.

In [27]:
print("=" * 80)
print("FINE-TUNING THE BEST XGBOOST CONFIGURATION")
print("=" * 80)

# Use the best feature subset found so far
selected_features = ranked_features[:40]

fine_tuning_configs = [

    {
        "n_estimators": 1100,
        "learning_rate": 0.015,
        "max_depth": 7,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0
    },

    {
        "n_estimators": 1300,
        "learning_rate": 0.015,
        "max_depth": 7,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0
    },

    {
        "n_estimators": 1000,
        "learning_rate": 0.02,
        "max_depth": 7,
        "min_child_weight": 3,
        "subsample": 0.85,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0
    },

    {
        "n_estimators": 1000,
        "learning_rate": 0.02,
        "max_depth": 7,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_lambda": 15.0
    },

    {
        "n_estimators": 1000,
        "learning_rate": 0.02,
        "max_depth": 7,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 10.0
    },

    {
        "n_estimators": 1000,
        "learning_rate": 0.02,
        "max_depth": 7,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 20.0
    },

    {
        "n_estimators": 1100,
        "learning_rate": 0.018,
        "max_depth": 7,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0
    },

    {
        "n_estimators": 1000,
        "learning_rate": 0.025,
        "max_depth": 7,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0
    },

    {
        "n_estimators": 1000,
        "learning_rate": 0.02,
        "max_depth": 8,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0
    },

    {
        "n_estimators": 1000,
        "learning_rate": 0.02,
        "max_depth": 7,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_lambda": 15.0
    }
]

fine_results = []
fine_models = {}

for i, config in enumerate(
    fine_tuning_configs,
    start=1
):

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        min_child_weight=config["min_child_weight"],

        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],

        reg_lambda=config["reg_lambda"],
        reg_alpha=0.0,

        scale_pos_weight=6.0,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train[selected_features],
        y_train,
        eval_set=[
            (
                X_validation[selected_features],
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = (
        model
        .predict_proba(
            X_validation[selected_features]
        )[:, 1]
    )

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.01
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    fine_models[i] = {
        "model": model,
        "features": selected_features
    }

    fine_results.append({
        "configuration": i,
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "min_child_weight": config["min_child_weight"],
        "subsample": config["subsample"],
        "colsample_bytree": config["colsample_bytree"],
        "reg_lambda": config["reg_lambda"],
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best_Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })

    print(
        f"Config {i:02d} | "
        f"PR-AUC={pr_auc:.4f} | "
        f"F1={best_f1:.4f} | "
        f"Threshold={best_threshold:.2f}"
    )


fine_tuning_df = (
    pd.DataFrame(fine_results)
    .sort_values(
        ["F1", "PR_AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTop fine-tuning configurations:")
display(
    fine_tuning_df.round(4)
)

print("\nBest configuration:")
display(
    fine_tuning_df.head(1).round(4)
)

FINE-TUNING THE BEST XGBOOST CONFIGURATION
Config 01 | PR-AUC=0.3394 | F1=0.3915 | Threshold=0.55
Config 02 | PR-AUC=0.3397 | F1=0.3879 | Threshold=0.50
Config 03 | PR-AUC=0.3386 | F1=0.3921 | Threshold=0.52
Config 04 | PR-AUC=0.3386 | F1=0.3882 | Threshold=0.55
Config 05 | PR-AUC=0.3390 | F1=0.3855 | Threshold=0.44
Config 06 | PR-AUC=0.3388 | F1=0.3913 | Threshold=0.50
Config 07 | PR-AUC=0.3382 | F1=0.3912 | Threshold=0.50
Config 08 | PR-AUC=0.3410 | F1=0.3932 | Threshold=0.49
Config 09 | PR-AUC=0.3376 | F1=0.3909 | Threshold=0.48
Config 10 | PR-AUC=0.3376 | F1=0.3907 | Threshold=0.52

Top fine-tuning configurations:


,configuration,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,8,1000,0.025,7,3,0.90,0.90,15.0,0.3410,0.8021,0.49,0.3304,0.4855,0.3932
1,3,1000,0.020,7,3,0.85,0.90,15.0,0.3386,0.8025,0.52,0.3394,0.4642,0.3921
2,1,1100,0.015,7,3,0.90,0.90,15.0,0.3394,0.8022,0.55,0.3569,0.4336,0.3915
3,6,1000,0.020,7,3,0.90,0.90,20.0,0.3388,0.8022,0.50,0.3258,0.4898,0.3913
4,7,1100,0.018,7,3,0.90,0.90,15.0,0.3382,0.8018,0.50,0.3269,0.4872,0.3912
5,9,1000,0.020,8,3,0.90,0.90,15.0,0.3376,0.8009,0.48,0.3308,0.4779,0.3909
6,10,1000,0.020,7,2,0.90,0.90,15.0,0.3376,0.8011,0.52,0.3401,0.4591,0.3907
7,4,1000,0.020,7,3,0.90,0.85,15.0,0.3386,0.8017,0.55,0.3555,0.4276,0.3882
8,2,1300,0.015,7,3,0.90,0.90,15.0,0.3397,0.8018,0.50,0.3249,0.4813,0.3879
9,5,1000,0.020,7,3,0.90,0.90,10.0,0.3390,0.8011,0.44,0.2972,0.5486,0.3855



Best configuration:


,configuration,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,8,1000,0.025,7,3,0.9,0.9,15.0,0.341,0.8021,0.49,0.3304,0.4855,0.3932


## 25. Fine-Grained Threshold Optimization

The classification threshold is refined using a smaller step size around the best-performing region.

Because the target class is imbalanced, the default threshold of 0.50 is not assumed to be optimal.

The threshold is selected exclusively from validation performance using the F1-score of the Late class.

In [28]:
print("=" * 80)
print("FINE-GRAINED THRESHOLD SEARCH")
print("=" * 80)

# Get the best configuration from the latest tuning results
best_row = (
    fine_tuning_df
    .iloc[0]
)

best_config_id = int(
    best_row["configuration"]
)

best_model_info = fine_models[
    best_config_id
]

best_model = best_model_info["model"]
selected_features = best_model_info["features"]

# Validation probabilities
best_prob = (
    best_model
    .predict_proba(
        X_validation[selected_features]
    )[:, 1]
)

# Search thresholds with a much smaller step
fine_thresholds = np.arange(
    0.35,
    0.66,
    0.001
)

fine_threshold_results = []

for threshold in fine_thresholds:

    predictions = (
        best_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    fine_threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

fine_threshold_df = pd.DataFrame(
    fine_threshold_results
)

best_fine_idx = (
    fine_threshold_df["f1"]
    .idxmax()
)

best_fine_threshold = (
    fine_threshold_df
    .loc[
        best_fine_idx,
        "threshold"
    ]
)

print(
    "Best threshold:",
    round(
        best_fine_threshold,
        3
    )
)

print(
    "Precision:",
    round(
        fine_threshold_df
        .loc[
            best_fine_idx,
            "precision"
        ],
        4
    )
)

print(
    "Recall:",
    round(
        fine_threshold_df
        .loc[
            best_fine_idx,
            "recall"
        ],
        4
    )
)

print(
    "F1:",
    round(
        fine_threshold_df
        .loc[
            best_fine_idx,
            "f1"
        ],
        4
    )
)

print("\nTop 20 thresholds:")
display(
    fine_threshold_df
    .sort_values(
        "f1",
        ascending=False
    )
    .head(20)
    .round(4)
)

print("\nPR-AUC:")
print(
    round(
        average_precision_score(
            y_validation,
            best_prob
        ),
        4
    )
)

print("\nROC-AUC:")
print(
    round(
        roc_auc_score(
            y_validation,
            best_prob
        ),
        4
    )
)

FINE-GRAINED THRESHOLD SEARCH
Best threshold: 0.491
Precision: 0.3314
Recall: 0.4847
F1: 0.3936

Top 20 thresholds:


,threshold,precision,recall,f1
141,0.491,0.3314,0.4847,0.3936
140,0.490,0.3304,0.4855,0.3932
139,0.489,0.3297,0.4864,0.3930
145,0.495,0.3329,0.4787,0.3927
144,0.494,0.3323,0.4796,0.3926
146,0.496,0.3335,0.4770,0.3926
175,0.525,0.3500,0.4463,0.3924
177,0.527,0.3505,0.4455,0.3923
137,0.487,0.3280,0.4881,0.3923
138,0.488,0.3287,0.4864,0.3923



PR-AUC:
0.341

ROC-AUC:
0.8021


## 26. Regularization and Class-Balance Tuning

Additional XGBoost configurations are tested by varying class weight, L1/L2 regularization, and `max_delta_step`.

These parameters are investigated to determine whether stronger regularization or improved control of the minority-class optimization can improve validation F1-score.

The test set remains untouched.

In [29]:
print("=" * 80)
print("XGBOOST REGULARIZATION AND CLASS-BALANCE TUNING")
print("=" * 80)

selected_features = ranked_features[:40]

configs = [
    {
        "weight": 5.0,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "max_delta_step": 0
    },
    {
        "weight": 6.0,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "max_delta_step": 0
    },
    {
        "weight": 7.0,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0,
        "max_delta_step": 0
    },
    {
        "weight": 5.0,
        "reg_lambda": 15.0,
        "reg_alpha": 0.5,
        "max_delta_step": 0
    },
    {
        "weight": 6.0,
        "reg_lambda": 15.0,
        "reg_alpha": 0.5,
        "max_delta_step": 0
    },
    {
        "weight": 7.0,
        "reg_lambda": 15.0,
        "reg_alpha": 0.5,
        "max_delta_step": 0
    },
    {
        "weight": 6.0,
        "reg_lambda": 15.0,
        "reg_alpha": 0.0,
        "max_delta_step": 1
    },
    {
        "weight": 6.0,
        "reg_lambda": 15.0,
        "reg_alpha": 0.0,
        "max_delta_step": 2
    },
    {
        "weight": 6.0,
        "reg_lambda": 20.0,
        "reg_alpha": 0.5,
        "max_delta_step": 1
    },
    {
        "weight": 7.0,
        "reg_lambda": 20.0,
        "reg_alpha": 0.5,
        "max_delta_step": 1
    }
]

results = []
models = {}

for i, config in enumerate(configs, start=1):

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=1000,
        learning_rate=0.025,

        max_depth=7,
        min_child_weight=3,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=config["reg_lambda"],
        reg_alpha=config["reg_alpha"],

        gamma=0.0,

        scale_pos_weight=config["weight"],
        max_delta_step=config["max_delta_step"],

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train[selected_features],
        y_train,
        eval_set=[
            (
                X_validation[selected_features],
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = (
        model
        .predict_proba(
            X_validation[selected_features]
        )[:, 1]
    )

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.001
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    models[i] = {
        "model": model,
        "features": selected_features
    }

    results.append({
        "configuration": i,
        "scale_pos_weight": config["weight"],
        "reg_lambda": config["reg_lambda"],
        "reg_alpha": config["reg_alpha"],
        "max_delta_step": config["max_delta_step"],
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best_Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })

    print(
        f"Config {i:02d} | "
        f"weight={config['weight']} | "
        f"lambda={config['reg_lambda']} | "
        f"alpha={config['reg_alpha']} | "
        f"delta={config['max_delta_step']} | "
        f"PR-AUC={pr_auc:.4f} | "
        f"F1={best_f1:.4f}"
    )


regularization_df = (
    pd.DataFrame(results)
    .sort_values(
        ["F1", "PR_AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nTop configurations:")
display(
    regularization_df.round(4)
)

XGBOOST REGULARIZATION AND CLASS-BALANCE TUNING
Config 01 | weight=5.0 | lambda=10.0 | alpha=0.0 | delta=0 | PR-AUC=0.3394 | F1=0.3893
Config 02 | weight=6.0 | lambda=10.0 | alpha=0.0 | delta=0 | PR-AUC=0.3406 | F1=0.3926
Config 03 | weight=7.0 | lambda=10.0 | alpha=0.0 | delta=0 | PR-AUC=0.3352 | F1=0.3920
Config 04 | weight=5.0 | lambda=15.0 | alpha=0.5 | delta=0 | PR-AUC=0.3410 | F1=0.3926
Config 05 | weight=6.0 | lambda=15.0 | alpha=0.5 | delta=0 | PR-AUC=0.3383 | F1=0.3906
Config 06 | weight=7.0 | lambda=15.0 | alpha=0.5 | delta=0 | PR-AUC=0.3405 | F1=0.3927
Config 07 | weight=6.0 | lambda=15.0 | alpha=0.0 | delta=1 | PR-AUC=0.3375 | F1=0.3894
Config 08 | weight=6.0 | lambda=15.0 | alpha=0.0 | delta=2 | PR-AUC=0.3397 | F1=0.3893
Config 09 | weight=6.0 | lambda=20.0 | alpha=0.5 | delta=1 | PR-AUC=0.3365 | F1=0.3911
Config 10 | weight=7.0 | lambda=20.0 | alpha=0.5 | delta=1 | PR-AUC=0.3371 | F1=0.3891

Top configurations:


,configuration,scale_pos_weight,reg_lambda,reg_alpha,max_delta_step,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,6,7.0,15.0,0.5,0,0.3405,0.8006,0.537,0.3393,0.4659,0.3927
1,2,6.0,10.0,0.0,0,0.3406,0.8028,0.458,0.3159,0.5187,0.3926
2,4,5.0,15.0,0.5,0,0.3410,0.8018,0.461,0.3327,0.4787,0.3926
3,3,7.0,10.0,0.0,0,0.3352,0.7997,0.525,0.3339,0.4744,0.3920
4,9,6.0,20.0,0.5,1,0.3365,0.8012,0.494,0.3260,0.4889,0.3911
5,5,6.0,15.0,0.5,0,0.3383,0.8015,0.486,0.3256,0.4881,0.3906
6,7,6.0,15.0,0.0,1,0.3375,0.8020,0.521,0.3433,0.4497,0.3894
7,1,5.0,10.0,0.0,0,0.3394,0.8026,0.428,0.3171,0.5043,0.3893
8,8,6.0,15.0,0.0,2,0.3397,0.8022,0.472,0.3177,0.5026,0.3893
9,10,7.0,20.0,0.5,1,0.3371,0.8010,0.530,0.3290,0.4761,0.3891


## 27. Compare XGBoost with Random Forest

XGBoost and Random Forest are compared using the strongest feature representation identified during the previous experiments.

Both models are evaluated using PR-AUC, ROC-AUC, precision, recall, and F1-score.

This comparison verifies whether the additional complexity of XGBoost provides a meaningful improvement over another non-linear tree-based model.

In [30]:
print("=" * 80)
print("MODEL COMPARISON — XGBOOST vs RANDOM FOREST")
print("=" * 80)

from sklearn.ensemble import RandomForestClassifier


# Use the current strongest feature subset
selected_features = ranked_features[:40]


models_to_test = {
    "XGBoost": xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=1000,
        learning_rate=0.025,
        max_depth=7,
        min_child_weight=3,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=15.0,
        reg_alpha=0.0,

        scale_pos_weight=6.0,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=5,
        min_samples_split=10,

        class_weight="balanced",

        random_state=42,
        n_jobs=-1
    )
}


model_results = []
comparison_models = {}

for model_name, model in models_to_test.items():

    print(f"\nTraining {model_name}...")

    model.fit(
        X_train[selected_features],
        y_train
    )

    probabilities = (
        model
        .predict_proba(
            X_validation[selected_features]
        )[:, 1]
    )

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.001
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    comparison_models[model_name] = {
        "model": model,
        "features": selected_features,
        "probabilities": probabilities
    }

    model_results.append({
        "Model": model_name,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best_Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })

    print(
        f"{model_name}: "
        f"PR-AUC={pr_auc:.4f} | "
        f"ROC-AUC={roc_auc:.4f} | "
        f"F1={best_f1:.4f} | "
        f"Threshold={best_threshold:.3f}"
    )


model_comparison_new = (
    pd.DataFrame(model_results)
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nModel comparison:")
display(
    model_comparison_new.round(4)
)

MODEL COMPARISON — XGBOOST vs RANDOM FOREST

Training XGBoost...
XGBoost: PR-AUC=0.3410 | ROC-AUC=0.8021 | F1=0.3936 | Threshold=0.491

Training Random Forest...
Random Forest: PR-AUC=0.3156 | ROC-AUC=0.7937 | F1=0.3717 | Threshold=0.630

Model comparison:


,Model,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,XGBoost,0.3410,0.8021,0.491,0.3314,0.4847,0.3936
1,Random Forest,0.3156,0.7937,0.630,0.3578,0.3867,0.3717


## 28. Extended Boosting-Round Experiment

An additional XGBoost experiment is performed using a larger number of boosting rounds to determine whether increasing the number of estimators improves validation performance.

This experiment does not use the test set and does not automatically replace the current best configuration.

In [31]:
print("=" * 80)
print("XGBOOST EARLY STOPPING")
print("=" * 80)

selected_features = ranked_features[:40]

early_stop_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",

    n_estimators=2000,
    learning_rate=0.025,

    max_depth=7,
    min_child_weight=3,

    subsample=0.90,
    colsample_bytree=0.90,

    reg_lambda=15.0,
    reg_alpha=0.0,

    scale_pos_weight=6.0,

    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

early_stop_model.fit(
    X_train[selected_features],
    y_train,
    eval_set=[
        (
            X_validation[selected_features],
            y_validation
        )
    ],
    verbose=False
)

best_iteration = (
    early_stop_model.best_iteration
    if hasattr(
        early_stop_model,
        "best_iteration"
    )
    else None
)

print(
    "Best iteration:",
    best_iteration
)

early_prob = (
    early_stop_model
    .predict_proba(
        X_validation[selected_features]
    )[:, 1]
)

early_pr_auc = (
    average_precision_score(
        y_validation,
        early_prob
    )
)

early_roc_auc = (
    roc_auc_score(
        y_validation,
        early_prob
    )
)

best_f1 = 0
best_threshold = 0
best_precision = 0
best_recall = 0

for threshold in np.arange(
    0.05,
    0.96,
    0.001
):

    predictions = (
        early_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
        best_precision = precision
        best_recall = recall

print("\nValidation Results")
print("=" * 60)

print(
    "PR-AUC:",
    round(
        early_pr_auc,
        4
    )
)

print(
    "ROC-AUC:",
    round(
        early_roc_auc,
        4
    )
)

print(
    "Best Threshold:",
    round(
        best_threshold,
        3
    )
)

print(
    "Precision:",
    round(
        best_precision,
        4
    )
)

print(
    "Recall:",
    round(
        best_recall,
        4
    )
)

print(
    "F1:",
    round(
        best_f1,
        4
    )
)

XGBOOST EARLY STOPPING
Best iteration: None

Validation Results
PR-AUC: 0.3313
ROC-AUC: 0.7959
Best Threshold: 0.425
Precision: 0.3239
Recall: 0.4957
F1: 0.3918


## 29. Geographic Representation Test

Different representations of the geographic information are compared to determine whether frequency-encoded city and ZIP-code features contribute useful information.

Three representations are evaluated: the full feature set, removal of both city and ZIP frequency features, and removal of ZIP frequency only.

The goal is to identify the representation that provides the best validation performance without increasing unnecessary feature complexity.

In [35]:
print("=" * 80)
print("GEOGRAPHIC REPRESENTATION TEST")
print("=" * 80)

# Start from the current best feature set
selected_features = ranked_features[:40]

# Remove frequency-encoded city and ZIP features
without_frequency_features = [
    feature
    for feature in selected_features
    if not feature.startswith(
        "frequency_customer_city"
    )
    and not feature.startswith(
        "frequency_customer_zip"
    )
]

# Also test removing only ZIP frequency
without_zip_frequency = [
    feature
    for feature in selected_features
    if not feature.startswith(
        "frequency_customer_zip"
    )
]

feature_sets = {
    "Top 40": selected_features,
    "Without city and ZIP frequency": without_frequency_features,
    "Without ZIP frequency": without_zip_frequency
}

geography_results = []
geography_models = {}

for feature_set_name, features in feature_sets.items():

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=1000,
        learning_rate=0.025,

        max_depth=7,
        min_child_weight=3,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=15.0,
        reg_alpha=0.0,

        scale_pos_weight=6.0,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train[features],
        y_train,
        eval_set=[
            (
                X_validation[features],
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = (
        model
        .predict_proba(
            X_validation[features]
        )[:, 1]
    )

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.001
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    geography_models[
        feature_set_name
    ] = {
        "model": model,
        "features": features,
        "probabilities": probabilities
    }

    geography_results.append({
        "Feature Set": feature_set_name,
        "Feature Count": len(features),
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best_Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })

    print(
        f"{feature_set_name}: "
        f"PR-AUC={pr_auc:.4f} | "
        f"ROC-AUC={roc_auc:.4f} | "
        f"F1={best_f1:.4f} | "
        f"Threshold={best_threshold:.3f}"
    )


geography_df = (
    pd.DataFrame(
        geography_results
    )
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nGeographic representation results:")
display(
    geography_df.round(4)
)

GEOGRAPHIC REPRESENTATION TEST
Top 40: PR-AUC=0.3410 | ROC-AUC=0.8021 | F1=0.3936 | Threshold=0.491
Without city and ZIP frequency: PR-AUC=0.3313 | ROC-AUC=0.7987 | F1=0.3942 | Threshold=0.528
Without ZIP frequency: PR-AUC=0.3385 | ROC-AUC=0.8024 | F1=0.4013 | Threshold=0.527

Geographic representation results:


,Feature Set,Feature Count,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,Without ZIP frequency,39,0.3385,0.8024,0.527,0.3555,0.4608,0.4013
1,Without city and ZIP frequency,38,0.3313,0.7987,0.528,0.3479,0.4549,0.3942
2,Top 40,40,0.3410,0.8021,0.491,0.3314,0.4847,0.3936


## 30. Class Weight Tuning on the Best Feature Set

After selecting the strongest geographic representation, different `scale_pos_weight` values are evaluated again using that feature set.

This experiment determines whether class weighting interacts with the selected feature representation and whether a different weight improves the F1-score.

In [36]:
print("=" * 80)
print("CLASS WEIGHT TUNING ON THE BEST FEATURE SET")
print("=" * 80)

# Use the best feature set found so far
selected_features = without_zip_frequency

weight_values = [
    4.0,
    5.0,
    6.0,
    7.0,
    8.0
]

weight_results = []
weight_models = {}

for weight in weight_values:

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=1000,
        learning_rate=0.025,

        max_depth=7,
        min_child_weight=3,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=15.0,
        reg_alpha=0.0,

        scale_pos_weight=weight,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train[selected_features],
        y_train,
        eval_set=[
            (
                X_validation[selected_features],
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = (
        model
        .predict_proba(
            X_validation[selected_features]
        )[:, 1]
    )

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.001
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    weight_models[weight] = {
        "model": model,
        "features": selected_features,
        "probabilities": probabilities
    }

    weight_results.append({
        "scale_pos_weight": weight,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best_Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1
    })

    print(
        f"Weight {weight:.1f} | "
        f"PR-AUC={pr_auc:.4f} | "
        f"ROC-AUC={roc_auc:.4f} | "
        f"F1={best_f1:.4f} | "
        f"Threshold={best_threshold:.3f}"
    )


weight_tuning_best_features_df = (
    pd.DataFrame(weight_results)
    .sort_values(
        ["F1", "PR_AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nResults:")
display(
    weight_tuning_best_features_df.round(4)
)

print("\nBest configuration:")
display(
    weight_tuning_best_features_df.head(1).round(4)
)

CLASS WEIGHT TUNING ON THE BEST FEATURE SET
Weight 4.0 | PR-AUC=0.3417 | ROC-AUC=0.8030 | F1=0.3949 | Threshold=0.421
Weight 5.0 | PR-AUC=0.3403 | ROC-AUC=0.8024 | F1=0.3944 | Threshold=0.428
Weight 6.0 | PR-AUC=0.3385 | ROC-AUC=0.8024 | F1=0.4013 | Threshold=0.527
Weight 7.0 | PR-AUC=0.3398 | ROC-AUC=0.8014 | F1=0.3930 | Threshold=0.571
Weight 8.0 | PR-AUC=0.3396 | ROC-AUC=0.8008 | F1=0.3960 | Threshold=0.595

Results:


,scale_pos_weight,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,6.0,0.3385,0.8024,0.527,0.3555,0.4608,0.4013
1,8.0,0.3396,0.8008,0.595,0.3592,0.4412,0.3960
2,4.0,0.3417,0.8030,0.421,0.3365,0.4779,0.3949
3,5.0,0.3403,0.8024,0.428,0.3162,0.5239,0.3944
4,7.0,0.3398,0.8014,0.571,0.3554,0.4395,0.3930



Best configuration:


,scale_pos_weight,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1
0,6.0,0.3385,0.8024,0.527,0.3555,0.4608,0.4013


## 31. Weak Feature Removal

The weakest features in the current model are identified using XGBoost feature importance.

Each weak feature is removed individually and the resulting model is evaluated on the validation set.

The purpose is to determine whether removing noisy or low-contribution features can improve model performance.

In [37]:
print("=" * 80)
print("WEAK FEATURE REMOVAL TEST")
print("=" * 80)

# Start from the best feature set found so far
selected_features = without_zip_frequency

# Train a reference model first
reference_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",

    n_estimators=1000,
    learning_rate=0.025,

    max_depth=7,
    min_child_weight=3,

    subsample=0.90,
    colsample_bytree=0.90,

    reg_lambda=15.0,
    reg_alpha=0.0,

    scale_pos_weight=6.0,

    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

reference_model.fit(
    X_train[selected_features],
    y_train,
    eval_set=[
        (
            X_validation[selected_features],
            y_validation
        )
    ],
    verbose=False
)

reference_prob = (
    reference_model
    .predict_proba(
        X_validation[selected_features]
    )[:, 1]
)

reference_pr_auc = (
    average_precision_score(
        y_validation,
        reference_prob
    )
)

reference_roc_auc = (
    roc_auc_score(
        y_validation,
        reference_prob
    )
)

reference_best_f1 = 0
reference_best_threshold = 0
reference_best_precision = 0
reference_best_recall = 0

for threshold in np.arange(
    0.05,
    0.96,
    0.001
):

    predictions = (
        reference_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    if f1 > reference_best_f1:
        reference_best_f1 = f1
        reference_best_threshold = threshold
        reference_best_precision = precision
        reference_best_recall = recall


# Rank the features in the current feature set
current_importance = pd.DataFrame({
    "feature": selected_features,
    "importance": reference_model.feature_importances_
}).sort_values(
    "importance",
    ascending=True
).reset_index(drop=True)

weak_features = (
    current_importance
    .head(10)["feature"]
    .tolist()
)

print("\nReference model")
print("-" * 60)

print(
    "Features:",
    len(selected_features)
)

print(
    "PR-AUC:",
    round(reference_pr_auc, 4)
)

print(
    "ROC-AUC:",
    round(reference_roc_auc, 4)
)

print(
    "Best threshold:",
    round(reference_best_threshold, 3)
)

print(
    "Precision:",
    round(reference_best_precision, 4)
)

print(
    "Recall:",
    round(reference_best_recall, 4)
)

print(
    "F1:",
    round(reference_best_f1, 4)
)

print("\nWeakest 10 features:")
display(
    current_importance.head(10).round(6)
)


# Test removing each weak feature
removal_results = []

for feature_to_remove in weak_features:

    features = [
        feature
        for feature in selected_features
        if feature != feature_to_remove
    ]

    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",

        n_estimators=1000,
        learning_rate=0.025,

        max_depth=7,
        min_child_weight=3,

        subsample=0.90,
        colsample_bytree=0.90,

        reg_lambda=15.0,
        reg_alpha=0.0,

        scale_pos_weight=6.0,

        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train[features],
        y_train,
        eval_set=[
            (
                X_validation[features],
                y_validation
            )
        ],
        verbose=False
    )

    probabilities = (
        model
        .predict_proba(
            X_validation[features]
        )[:, 1]
    )

    pr_auc = average_precision_score(
        y_validation,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_validation,
        probabilities
    )

    best_f1 = 0
    best_threshold = 0
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(
        0.05,
        0.96,
        0.001
    ):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    removal_results.append({
        "Removed Feature": feature_to_remove,
        "PR_AUC": pr_auc,
        "ROC_AUC": roc_auc,
        "Best_Threshold": best_threshold,
        "Precision": best_precision,
        "Recall": best_recall,
        "F1": best_f1,
        "F1_Change": best_f1 - reference_best_f1
    })


removal_df = (
    pd.DataFrame(removal_results)
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFeature removal results:")
display(
    removal_df.round(4)
)

WEAK FEATURE REMOVAL TEST

Reference model
------------------------------------------------------------
Features: 39
PR-AUC: 0.3385
ROC-AUC: 0.8024
Best threshold: 0.527
Precision: 0.3555
Recall: 0.4608
F1: 0.4013

Weakest 10 features:


,feature,importance
0,onehot_customer_state_RN,0.008657
1,num_number_of_payments,0.011383
2,num_total_price,0.012473
3,onehot_customer_state_PB,0.012976
4,num_total_payment,0.013139
5,onehot_main_payment_type_boleto,0.013146
6,num_total_product_volume,0.013432
7,num_total_freight,0.013667
8,num_total_product_weight,0.013921
9,frequency_customer_city_frequency,0.014225



Feature removal results:


,Removed Feature,PR_AUC,ROC_AUC,Best_Threshold,Precision,Recall,F1,F1_Change
0,num_total_freight,0.3428,0.8060,0.505,0.3423,0.4864,0.4018,0.0005
1,num_number_of_payments,0.3394,0.8022,0.542,0.3646,0.4404,0.3989,-0.0024
2,num_total_payment,0.3432,0.8034,0.507,0.3415,0.4736,0.3969,-0.0045
3,num_total_product_weight,0.3406,0.8013,0.492,0.3281,0.4991,0.3959,-0.0054
4,num_total_product_volume,0.3373,0.8006,0.481,0.3231,0.5102,0.3956,-0.0057
5,onehot_main_payment_type_boleto,0.3379,0.8012,0.511,0.3418,0.4685,0.3953,-0.0061
6,onehot_customer_state_RN,0.3409,0.8030,0.526,0.3486,0.4540,0.3944,-0.0070
7,frequency_customer_city_frequency,0.3313,0.7987,0.528,0.3479,0.4549,0.3942,-0.0071
8,num_total_price,0.3398,0.8025,0.486,0.3263,0.4974,0.3941,-0.0073
9,onehot_customer_state_PB,0.3388,0.8032,0.497,0.3302,0.4804,0.3914,-0.0099


## 32. Build the Best Candidate Model Found So Far

After the previous feature and model experiments, the strongest feature representation and XGBoost configuration are combined into a final candidate model.

The selected candidate removes ZIP frequency encoding and the `total_freight` feature, while retaining the remaining informative features.

The candidate model is evaluated carefully on the validation set before the final test evaluation.

In [38]:
print("=" * 80)
print("FINAL CANDIDATE MODEL TEST")
print("=" * 80)

# Start from the best feature set found so far
candidate_features = [
    feature
    for feature in without_zip_frequency
    if feature != "num_total_freight"
]

candidate_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",

    n_estimators=1000,
    learning_rate=0.025,

    max_depth=7,
    min_child_weight=3,

    subsample=0.90,
    colsample_bytree=0.90,

    reg_lambda=15.0,
    reg_alpha=0.0,

    scale_pos_weight=6.0,

    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

candidate_model.fit(
    X_train[candidate_features],
    y_train,
    eval_set=[
        (
            X_validation[candidate_features],
            y_validation
        )
    ],
    verbose=False
)

candidate_prob = (
    candidate_model
    .predict_proba(
        X_validation[candidate_features]
    )[:, 1]
)

candidate_pr_auc = average_precision_score(
    y_validation,
    candidate_prob
)

candidate_roc_auc = roc_auc_score(
    y_validation,
    candidate_prob
)

# Fine threshold search
candidate_thresholds = np.arange(
    0.35,
    0.65,
    0.001
)

candidate_threshold_results = []

for threshold in candidate_thresholds:

    predictions = (
        candidate_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )

    candidate_threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

candidate_threshold_df = pd.DataFrame(
    candidate_threshold_results
)

best_candidate_row = (
    candidate_threshold_df
    .loc[
        candidate_threshold_df["f1"].idxmax()
    ]
)

print("\nCandidate model results")
print("=" * 60)

print(
    "Feature count:",
    len(candidate_features)
)

print(
    "PR-AUC:",
    round(candidate_pr_auc, 4)
)

print(
    "ROC-AUC:",
    round(candidate_roc_auc, 4)
)

print(
    "Best threshold:",
    round(
        best_candidate_row["threshold"],
        3
    )
)

print(
    "Precision:",
    round(
        best_candidate_row["precision"],
        4
    )
)

print(
    "Recall:",
    round(
        best_candidate_row["recall"],
        4
    )
)

print(
    "F1:",
    round(
        best_candidate_row["f1"],
        4
    )
)

print("\nTop 20 thresholds:")
display(
    candidate_threshold_df
    .sort_values(
        "f1",
        ascending=False
    )
    .head(20)
    .round(4)
)

print("\nCandidate features:")
for i, feature in enumerate(
    candidate_features,
    start=1
):
    print(
        f"{i:02d}. {feature}"
    )

FINAL CANDIDATE MODEL TEST

Candidate model results
Feature count: 38
PR-AUC: 0.3428
ROC-AUC: 0.806
Best threshold: 0.505
Precision: 0.3423
Recall: 0.4864
F1: 0.4018

Top 20 thresholds:


,threshold,precision,recall,f1
155,0.505,0.3423,0.4864,0.4018
156,0.506,0.3425,0.4855,0.4017
154,0.504,0.3415,0.4872,0.4015
157,0.507,0.3428,0.4838,0.4013
159,0.509,0.3433,0.4813,0.4007
158,0.508,0.3428,0.4821,0.4007
184,0.534,0.3584,0.4540,0.4006
176,0.526,0.3538,0.4617,0.4006
160,0.510,0.3435,0.4804,0.4006
183,0.533,0.3571,0.4557,0.4004



Candidate features:
01. onehot_customer_state_SP
02. onehot_customer_state_RJ
03. num_purchase_year
04. num_purchase_month
05. num_customer_seller_same_state
06. num_estimated_delivery_days
07. onehot_customer_state_MG
08. onehot_customer_state_PR
09. num_customer_seller_distance_km
10. num_number_of_sellers
11. onehot_customer_state_BA
12. onehot_customer_state_ES
13. onehot_customer_state_AL
14. num_number_of_items
15. onehot_customer_state_SC
16. onehot_customer_state_CE
17. num_purchase_dayofmonth
18. onehot_customer_state_RS
19. onehot_main_payment_type_credit_card
20. onehot_main_payment_type_boleto
21. num_total_product_weight
22. onehot_customer_state_MA
23. onehot_customer_state_DF
24. onehot_customer_state_MS
25. num_number_of_product_categories
26. onehot_customer_state_PA
27. onehot_customer_state_PE
28. frequency_customer_city_frequency
29. num_total_product_volume
30. onehot_customer_state_GO
31. num_is_holiday
32. onehot_customer_state_MT
33. num_total_price
34. num_num

## 33. Final Validation Check

The best candidate model is checked again around the optimal threshold to verify that its F1-score is not dependent on a single narrow threshold value.

A stable range of nearby thresholds provides additional confidence that the selected classification threshold is reasonable.

The test set remains untouched at this stage.

In [39]:
print("=" * 80)
print("FINAL VALIDATION CHECK")
print("=" * 80)

# Use the current best candidate model
final_candidate_features = candidate_features
final_candidate_model = candidate_model
final_candidate_prob = candidate_prob

# Calculate final validation metrics
final_validation_pr_auc = average_precision_score(
    y_validation,
    final_candidate_prob
)

final_validation_roc_auc = roc_auc_score(
    y_validation,
    final_candidate_prob
)

threshold_check = []

for threshold in np.arange(
    0.48,
    0.541,
    0.001
):

    predictions = (
        final_candidate_prob >= threshold
    ).astype(int)

    threshold_check.append({
        "threshold": threshold,

        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        )
    })

threshold_check_df = pd.DataFrame(
    threshold_check
)

best_validation_row = (
    threshold_check_df
    .loc[
        threshold_check_df["f1"].idxmax()
    ]
)

print("\nBest validation point")
print("=" * 60)

print(
    "PR-AUC:",
    round(
        final_validation_pr_auc,
        4
    )
)

print(
    "ROC-AUC:",
    round(
        final_validation_roc_auc,
        4
    )
)

print(
    "Best threshold:",
    round(
        best_validation_row["threshold"],
        3
    )
)

print(
    "Precision:",
    round(
        best_validation_row["precision"],
        4
    )
)

print(
    "Recall:",
    round(
        best_validation_row["recall"],
        4
    )
)

print(
    "F1:",
    round(
        best_validation_row["f1"],
        4
    )
)

print("\nThreshold stability:")
display(
    threshold_check_df
    .sort_values(
        "f1",
        ascending=False
    )
    .head(15)
    .round(4)
)

print("\nFinal candidate feature count:")
print(
    len(final_candidate_features)
)

FINAL VALIDATION CHECK

Best validation point
PR-AUC: 0.3428
ROC-AUC: 0.806
Best threshold: 0.505
Precision: 0.3423
Recall: 0.4864
F1: 0.4018

Threshold stability:


,threshold,precision,recall,f1
25,0.505,0.3423,0.4864,0.4018
26,0.506,0.3425,0.4855,0.4017
24,0.504,0.3415,0.4872,0.4015
27,0.507,0.3428,0.4838,0.4013
29,0.509,0.3433,0.4813,0.4007
28,0.508,0.3428,0.4821,0.4007
54,0.534,0.3584,0.4540,0.4006
46,0.526,0.3538,0.4617,0.4006
30,0.510,0.3435,0.4804,0.4006
53,0.533,0.3571,0.4557,0.4004



Final candidate feature count:
38


## 34. Ensemble Model

Several XGBoost models with slightly different configurations are combined to determine whether their predicted probabilities can provide more stable and accurate classification than a single model.

The ensemble combines the predicted probabilities of three independently trained XGBoost models using a weighted average.

The ensemble weights and classification threshold are selected using the validation set only. The test set remains completely unseen during this process.

The best ensemble configuration achieves a validation F1-score of 0.4061, slightly improving upon the best single XGBoost model with an F1-score of 0.4046.

The selected ensemble configuration is:

- 70%: Best-F1 XGBoost model
- 20%: Best-PR-AUC XGBoost model
- 10%: XGBoost model with `scale_pos_weight = 5.25`
- Classification threshold: 0.530

This ensemble is selected as the final model for the final test evaluation.

In [56]:
# ============================================================
# EXPERIMENT — FINE-TUNE CLASS WEIGHT
# ============================================================
# We keep the exact protected 38-feature set.
# Only scale_pos_weight is changed.
#
# The protected model itself is NOT modified.
# Validation is used only for comparison.
# Test is NOT touched.
# ============================================================

weight_values = [
    5.25,
    5.50,
    5.75,
    6.00,
    6.25,
    6.50,
    6.75
]

weight_results = []

for weight in weight_values:

    print(f"\nTraining model with scale_pos_weight = {weight}")

    model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.025,
        max_depth=7,
        min_child_weight=3,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_lambda=15,
        reg_alpha=0,
        scale_pos_weight=weight,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[candidate_features],
        y_train
    )

    prob = model.predict_proba(
        X_validation[candidate_features]
    )[:, 1]

    pr_auc = average_precision_score(
        y_validation,
        prob
    )

    roc_auc = roc_auc_score(
        y_validation,
        prob
    )

    best_f1 = 0
    best_threshold = None
    best_precision = 0
    best_recall = 0

    for threshold in np.arange(0.45, 0.601, 0.001):

        pred = (
            prob >= threshold
        ).astype(int)

        precision = precision_score(
            y_validation,
            pred,
            zero_division=0
        )

        recall = recall_score(
            y_validation,
            pred,
            zero_division=0
        )

        f1 = f1_score(
            y_validation,
            pred,
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    weight_results.append({
        "scale_pos_weight": weight,
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "threshold": best_threshold,
        "precision": best_precision,
        "recall": best_recall,
        "F1": best_f1
    })


weight_results_df = (
    pd.DataFrame(weight_results)
    .sort_values("F1", ascending=False)
)

print("\n" + "=" * 80)
print("FINE CLASS-WEIGHT TUNING RESULTS")
print("=" * 80)

display(
    weight_results_df.round(4)
)


Training model with scale_pos_weight = 5.25

Training model with scale_pos_weight = 5.5

Training model with scale_pos_weight = 5.75

Training model with scale_pos_weight = 6.0

Training model with scale_pos_weight = 6.25

Training model with scale_pos_weight = 6.5

Training model with scale_pos_weight = 6.75

FINE CLASS-WEIGHT TUNING RESULTS


,scale_pos_weight,PR-AUC,ROC-AUC,threshold,precision,recall,F1
4,6.25,0.3416,0.8063,0.539,0.3596,0.4625,0.4046
0,5.25,0.3442,0.8061,0.517,0.3654,0.4463,0.4018
3,6.00,0.3428,0.8060,0.505,0.3423,0.4864,0.4018
5,6.50,0.3413,0.8064,0.540,0.3540,0.4617,0.4007
1,5.50,0.3423,0.8061,0.494,0.3422,0.4813,0.4000
2,5.75,0.3421,0.8059,0.516,0.3517,0.4617,0.3993
6,6.75,0.3438,0.8055,0.516,0.3337,0.4923,0.3978


In [57]:
# ============================================================
# EXPERIMENT — TREE STRUCTURE TUNING
# ============================================================
# We keep the protected 38-feature representation.
# Only max_depth and min_child_weight are changed.
#
# Current best:
# F1 = 0.4046
# scale_pos_weight = 6.25
#
# The protected model is NOT modified.
# The test set is NOT used.
# ============================================================

depth_values = [6, 7, 8, 9]
min_child_values = [1, 3, 5]

tree_tuning_results = []

for depth in depth_values:

    for min_child in min_child_values:

        print(
            f"\nTraining: "
            f"max_depth={depth}, "
            f"min_child_weight={min_child}"
        )

        model = xgb.XGBClassifier(
            n_estimators=1000,
            learning_rate=0.025,
            max_depth=depth,
            min_child_weight=min_child,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_lambda=15,
            reg_alpha=0,
            scale_pos_weight=6.25,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train[candidate_features],
            y_train
        )

        prob = model.predict_proba(
            X_validation[candidate_features]
        )[:, 1]

        pr_auc = average_precision_score(
            y_validation,
            prob
        )

        roc_auc = roc_auc_score(
            y_validation,
            prob
        )

        best_f1 = 0
        best_threshold = None
        best_precision = 0
        best_recall = 0

        # Fine threshold search
        for threshold in np.arange(0.45, 0.601, 0.001):

            pred = (
                prob >= threshold
            ).astype(int)

            precision = precision_score(
                y_validation,
                pred,
                zero_division=0
            )

            recall = recall_score(
                y_validation,
                pred,
                zero_division=0
            )

            f1 = f1_score(
                y_validation,
                pred,
                zero_division=0
            )

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
                best_precision = precision
                best_recall = recall

        tree_tuning_results.append({
            "max_depth": depth,
            "min_child_weight": min_child,
            "PR-AUC": pr_auc,
            "ROC-AUC": roc_auc,
            "threshold": best_threshold,
            "precision": best_precision,
            "recall": best_recall,
            "F1": best_f1
        })


tree_tuning_df = (
    pd.DataFrame(tree_tuning_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("TREE STRUCTURE TUNING RESULTS")
print("=" * 80)

display(tree_tuning_df.round(4))

print("\nCurrent best F1: 0.4046")
print(
    "Best experimental F1:",
    round(tree_tuning_df.iloc[0]["F1"], 4)
)


Training: max_depth=6, min_child_weight=1

Training: max_depth=6, min_child_weight=3

Training: max_depth=6, min_child_weight=5

Training: max_depth=7, min_child_weight=1

Training: max_depth=7, min_child_weight=3

Training: max_depth=7, min_child_weight=5

Training: max_depth=8, min_child_weight=1

Training: max_depth=8, min_child_weight=3

Training: max_depth=8, min_child_weight=5

Training: max_depth=9, min_child_weight=1

Training: max_depth=9, min_child_weight=3

Training: max_depth=9, min_child_weight=5

TREE STRUCTURE TUNING RESULTS


,max_depth,min_child_weight,PR-AUC,ROC-AUC,threshold,precision,recall,F1
0,7,3,0.3416,0.8063,0.539,0.3596,0.4625,0.4046
1,9,5,0.3435,0.8072,0.504,0.3666,0.4506,0.4043
2,7,1,0.3405,0.8056,0.517,0.3477,0.4753,0.4016
3,9,3,0.3454,0.8068,0.473,0.3528,0.4642,0.4009
4,6,1,0.3424,0.8038,0.555,0.3556,0.4583,0.4004
5,8,1,0.3401,0.8037,0.480,0.3420,0.4830,0.4004
6,9,1,0.3425,0.8036,0.450,0.3460,0.4744,0.4001
7,7,5,0.3401,0.8059,0.535,0.3507,0.4634,0.3993
8,8,3,0.3428,0.8056,0.464,0.3300,0.5051,0.3992
9,8,5,0.3432,0.8067,0.524,0.3593,0.4472,0.3985



Current best F1: 0.4046
Best experimental F1: 0.4046


In [68]:
# ============================================================
# FINAL EXPERIMENT — XGBOOST ENSEMBLE
# ============================================================
# We combine several independent XGBoost models.
#
# Important:
# - This is a separate experiment.
# - Existing models are NOT modified.
# - The test set is NOT used.
# - All ensemble decisions are based on validation data only.
# ============================================================

ensemble_models = {}
ensemble_probabilities = {}

ensemble_configs = {
    "best_f1": {
        "max_depth": 7,
        "min_child_weight": 3,
        "scale_pos_weight": 6.25
    },

    "best_pr_auc": {
        "max_depth": 9,
        "min_child_weight": 3,
        "scale_pos_weight": 6.25
    },

    "weight_5_25": {
        "max_depth": 7,
        "min_child_weight": 3,
        "scale_pos_weight": 5.25
    }
}


for model_name, config in ensemble_configs.items():

    print(
        f"\nTraining {model_name}: "
        f"depth={config['max_depth']}, "
        f"child={config['min_child_weight']}, "
        f"weight={config['scale_pos_weight']}"
    )

    model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.025,
        max_depth=config["max_depth"],
        min_child_weight=config["min_child_weight"],
        subsample=0.90,
        colsample_bytree=0.90,
        reg_lambda=15,
        reg_alpha=0,
        scale_pos_weight=config["scale_pos_weight"],
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[candidate_features],
        y_train
    )

    prob = model.predict_proba(
        X_validation[candidate_features]
    )[:, 1]

    ensemble_models[model_name] = model
    ensemble_probabilities[model_name] = prob

    print(f"{model_name} trained successfully.")


print("\nAll ensemble models trained successfully.")


Training best_f1: depth=7, child=3, weight=6.25
best_f1 trained successfully.

Training best_pr_auc: depth=9, child=3, weight=6.25
best_pr_auc trained successfully.

Training weight_5_25: depth=7, child=3, weight=5.25
weight_5_25 trained successfully.

All ensemble models trained successfully.


In [69]:
# ============================================================
# ENSEMBLE WEIGHT SEARCH
# ============================================================
# We combine model probabilities using weighted averages.
# The weights are selected using validation data only.
# ============================================================

p1 = ensemble_probabilities["best_f1"]
p2 = ensemble_probabilities["best_pr_auc"]
p3 = ensemble_probabilities["weight_5_25"]


ensemble_results = []

# Try different weights for the three models.
# The weights always sum to 1.
for w1 in np.arange(0, 1.01, 0.10):

    for w2 in np.arange(0, 1.01 - w1, 0.10):

        w3 = 1.0 - w1 - w2

        if w3 < -1e-9:
            continue

        ensemble_prob = (
            w1 * p1 +
            w2 * p2 +
            w3 * p3
        )

        pr_auc = average_precision_score(
            y_validation,
            ensemble_prob
        )

        roc_auc = roc_auc_score(
            y_validation,
            ensemble_prob
        )

        # Find the threshold that maximizes F1
        best_f1 = 0
        best_threshold = 0
        best_precision = 0
        best_recall = 0

        for threshold in np.arange(0.40, 0.651, 0.001):

            pred = (
                ensemble_prob >= threshold
            ).astype(int)

            precision = precision_score(
                y_validation,
                pred,
                zero_division=0
            )

            recall = recall_score(
                y_validation,
                pred,
                zero_division=0
            )

            f1 = f1_score(
                y_validation,
                pred,
                zero_division=0
            )

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
                best_precision = precision
                best_recall = recall

        ensemble_results.append({
            "weight_best_f1": w1,
            "weight_best_pr_auc": w2,
            "weight_5_25": w3,
            "PR-AUC": pr_auc,
            "ROC-AUC": roc_auc,
            "threshold": best_threshold,
            "precision": best_precision,
            "recall": best_recall,
            "F1": best_f1
        })


ensemble_results_df = (
    pd.DataFrame(ensemble_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)


print("=" * 80)
print("TOP ENSEMBLE CONFIGURATIONS")
print("=" * 80)

display(
    ensemble_results_df.head(15).round(4)
)

print("\nCurrent best single model F1: 0.4046")
print(
    "Best ensemble F1:",
    round(ensemble_results_df.iloc[0]["F1"], 4)
)

TOP ENSEMBLE CONFIGURATIONS


,weight_best_f1,weight_best_pr_auc,weight_5_25,PR-AUC,ROC-AUC,threshold,precision,recall,F1
0,0.7,0.2,0.1,0.3440,0.8073,0.530,0.3656,0.4566,0.4061
1,0.4,0.2,0.4,0.3451,0.8073,0.522,0.3682,0.4523,0.4060
2,0.8,0.2,-0.0,0.3436,0.8072,0.527,0.3615,0.4625,0.4058
3,0.6,0.2,0.2,0.3443,0.8073,0.527,0.3651,0.4566,0.4058
4,0.6,0.1,0.3,0.3438,0.8070,0.517,0.3581,0.4676,0.4056
5,0.8,0.1,0.1,0.3432,0.8069,0.532,0.3627,0.4600,0.4056
6,0.7,0.1,0.2,0.3435,0.8069,0.522,0.3587,0.4659,0.4053
7,0.5,0.2,0.3,0.3448,0.8073,0.519,0.3627,0.4591,0.4053
8,0.2,0.7,0.1,0.3461,0.8077,0.445,0.3350,0.5128,0.4053
9,0.5,0.3,0.2,0.3452,0.8076,0.518,0.3633,0.4574,0.4050



Current best single model F1: 0.4046
Best ensemble F1: 0.4061


## 35. Save the Final Ensemble Configuration

The ensemble configuration that achieved the best validation F1-score is saved for final evaluation.

The individual XGBoost models are kept unchanged. The selected weights and classification threshold are also stored so that the exact same configuration can be applied to the unseen test set.

No further tuning will be performed after this step.

In [71]:
# ============================================================
# SAVE FINAL ENSEMBLE CONFIGURATION
# ============================================================
# This is the best validation configuration found so far.
# We save the individual models, ensemble weights, and threshold.
# The original single-model experiments remain untouched.
# ============================================================

output_dir = "../artifacts/random_split/notebook6"
os.makedirs(output_dir, exist_ok=True)


# Best ensemble weights selected on validation
final_ensemble_weights = {
    "best_f1": 0.70,
    "best_pr_auc": 0.20,
    "weight_5_25": 0.10
}

# Threshold selected on validation
final_ensemble_threshold = 0.530


# Save the three XGBoost models
joblib.dump(
    ensemble_models["best_f1"],
    os.path.join(
        output_dir,
        "ensemble_best_f1_model1.joblib"
    )
)

joblib.dump(
    ensemble_models["best_pr_auc"],
    os.path.join(
        output_dir,
        "ensemble_best_pr_auc_model1.joblib"
    )
)

joblib.dump(
    ensemble_models["weight_5_25"],
    os.path.join(
        output_dir,
        "ensemble_weight_5_25_model1.joblib"
    )
)


# Save ensemble configuration
ensemble_config = {
    "model_type": "weighted_xgboost_ensemble",
    "weights": final_ensemble_weights,
    "threshold": final_ensemble_threshold,
    "feature_count": len(candidate_features),
    "validation_f1": 0.4061,
    "validation_pr_auc": 0.3440,
    "validation_roc_auc": 0.8073
}

with open(
    os.path.join(
        output_dir,
        "final_ensemble_configuration1.json"
    ),
    "w"
) as f:
    json.dump(
        ensemble_config,
        f,
        indent=4
    )


print("=" * 80)
print("FINAL ENSEMBLE SAVED")
print("=" * 80)

print("Validation F1    : 0.4061")
print("Validation PR-AUC: 0.3440")
print("Validation ROC-AUC: 0.8073")
print("Threshold        : 0.530")

print("\nWeights:")
for name, weight in final_ensemble_weights.items():
    print(f"{name}: {weight:.2f}")

print("\n Final ensemble configuration saved successfully.")

FINAL ENSEMBLE SAVED
Validation F1    : 0.4061
Validation PR-AUC: 0.3440
Validation ROC-AUC: 0.8073
Threshold        : 0.530

Weights:
best_f1: 0.70
best_pr_auc: 0.20
weight_5_25: 0.10

 Final ensemble configuration saved successfully.


## 36. Final Test Evaluation

The final ensemble is now evaluated on the unseen test set.

The test set has not been used for feature selection, hyperparameter tuning, ensemble weighting, or threshold selection.

The final prediction is generated by combining the probabilities of the three selected XGBoost models using the weights determined from the validation set. The fixed classification threshold of 0.530 is then applied.

This is the first and only evaluation of the final model on the test set.

In [72]:
# ============================================================
# FINAL TEST EVALUATION — ENSEMBLE
# ============================================================
# IMPORTANT:
# This is the final test evaluation.
# Do NOT change the weights or threshold after seeing the result.
# ============================================================


# ------------------------------------------------------------
# Generate test probabilities from each ensemble model
# ------------------------------------------------------------

test_prob_best_f1 = (
    ensemble_models["best_f1"]
    .predict_proba(X_test[candidate_features])[:, 1]
)

test_prob_best_pr_auc = (
    ensemble_models["best_pr_auc"]
    .predict_proba(X_test[candidate_features])[:, 1]
)

test_prob_weight_5_25 = (
    ensemble_models["weight_5_25"]
    .predict_proba(X_test[candidate_features])[:, 1]
)


# ------------------------------------------------------------
# Weighted ensemble probability
# ------------------------------------------------------------

final_test_probability = (
    0.70 * test_prob_best_f1
    + 0.20 * test_prob_best_pr_auc
    + 0.10 * test_prob_weight_5_25
)


# ------------------------------------------------------------
# Apply the validation-selected threshold
# ------------------------------------------------------------

final_test_prediction = (
    final_test_probability >= final_ensemble_threshold
).astype(int)


# ------------------------------------------------------------
# Calculate final metrics
# ------------------------------------------------------------

final_test_accuracy = accuracy_score(
    y_test,
    final_test_prediction
)

final_test_precision = precision_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_recall = recall_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_f1 = f1_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_test_roc_auc = roc_auc_score(
    y_test,
    final_test_probability
)

final_test_pr_auc = average_precision_score(
    y_test,
    final_test_probability
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

final_test_cm = confusion_matrix(
    y_test,
    final_test_prediction
)


# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print("=" * 80)
print("FINAL TEST RESULTS — WEIGHTED XGBOOST ENSEMBLE")
print("=" * 80)

print(f"Threshold : {final_ensemble_threshold:.3f}")
print(f"Accuracy  : {final_test_accuracy:.4f}")
print(f"Precision : {final_test_precision:.4f}")
print(f"Recall    : {final_test_recall:.4f}")
print(f"F1        : {final_test_f1:.4f}")
print(f"ROC-AUC   : {final_test_roc_auc:.4f}")
print(f"PR-AUC    : {final_test_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(final_test_cm)


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        final_test_prediction,
        target_names=["On-Time", "Late"],
        zero_division=0
    )
)

FINAL TEST RESULTS — WEIGHTED XGBOOST ENSEMBLE
Threshold : 0.530
Accuracy  : 0.8856
Precision : 0.3408
Recall    : 0.4387
F1        : 0.3836
ROC-AUC   : 0.8007
PR-AUC    : 0.3242

Confusion Matrix:
[[12301   996]
 [  659   515]]

Classification Report:
              precision    recall  f1-score   support

     On-Time       0.95      0.93      0.94     13297
        Late       0.34      0.44      0.38      1174

    accuracy                           0.89     14471
   macro avg       0.64      0.68      0.66     14471
weighted avg       0.90      0.89      0.89     14471



## 37. Final Model Performance

The weighted XGBoost ensemble was selected based on validation performance and evaluated once on the unseen test set.

The final model combines the predicted probabilities of three XGBoost classifiers using weights selected from the validation set, together with the classification threshold selected during validation.

The final evaluation reports Accuracy, Precision, Recall, F1-score, ROC-AUC, and PR-AUC, with particular emphasis on F1-score and PR-AUC because the target variable is imbalanced.

The test set was not used during model selection, hyperparameter tuning, ensemble weighting, or threshold selection. No further tuning was performed after the final test evaluation.


In [74]:
# ============================================================
# SAVE FINAL TEST RESULTS
# ============================================================
# The test set has now been evaluated once.
# These results are final and will not be used for further tuning.
# ============================================================

final_results_summary = {
    "model": "Weighted XGBoost Ensemble",
    "feature_count": len(candidate_features),

    "ensemble_weights": {
        "best_f1": 0.70,
        "best_pr_auc": 0.20,
        "weight_5_25": 0.10
    },

    "threshold": 0.530,

    # Validation performance
    "validation_f1": 0.4061,
    "validation_pr_auc": 0.3440,
    "validation_roc_auc": 0.8073,

    # Final test performance
    "test_accuracy": final_test_accuracy,
    "test_precision": final_test_precision,
    "test_recall": final_test_recall,
    "test_f1": final_test_f1,
    "test_roc_auc": final_test_roc_auc,
    "test_pr_auc": final_test_pr_auc,

    # Confusion matrix
    "confusion_matrix": final_test_cm.tolist()
}


with open(
    os.path.join(
        output_dir,
        "results_summary1.json"
    ),
    "w"
) as f:

    json.dump(
        final_results_summary,
        f,
        indent=4
    )


print("=" * 80)
print("FINAL RESULTS SAVED")
print("=" * 80)

print(
    "Saved:",
    os.path.join(
        output_dir,
        "results_summary1.json"
    )
)

print("\n Notebook 6 final results saved successfully.")

FINAL RESULTS SAVED
Saved: ../artifacts/random_split/notebook6\results_summary1.json

 Notebook 6 final results saved successfully.


## Final Conclusion

This notebook developed and evaluated machine learning models for predicting late deliveries in the Olist Brazilian e-commerce dataset.

The target variable was highly imbalanced, with late deliveries representing a minority of the orders. Therefore, model evaluation focused primarily on F1-score, Precision, Recall, ROC-AUC, and PR-AUC rather than accuracy alone.

A simple majority-class baseline and Logistic Regression were first used as reference models. XGBoost was then investigated through multiple stages of feature selection, class-weight tuning, tree-structure tuning, threshold optimization, regularization, and alternative model comparisons.

The experiments showed that the strongest individual XGBoost configuration used a reduced feature set of 38 features. The best single-model validation performance reached an F1-score of 0.4046.

To further improve the prediction stability, three XGBoost models with slightly different configurations were combined using a weighted probability ensemble. The final ensemble assigned 70% weight to the best-F1 model, 20% to the best-PR-AUC model, and 10% to the model using `scale_pos_weight = 5.25`.

The ensemble achieved a validation F1-score of 0.4061, with a validation PR-AUC of 0.3440 and ROC-AUC of 0.8073. The classification threshold of 0.530 was selected exclusively using the validation set.

The final ensemble was then evaluated once on the unseen test set.

The final test performance was:

* Accuracy: 0.8856
* Precision: 0.3408
* Recall: 0.4387
* F1-score: 0.3836
* ROC-AUC: 0.8007
* PR-AUC: 0.3242

The model correctly identified 43.87% of the actual late deliveries while maintaining a precision of 34.08% for the Late class.

The decrease from validation F1-score (0.4061) to test F1-score (0.3836) indicates some reduction in generalization on unseen data. However, the ROC-AUC remained close between validation (0.8073) and test (0.8007), indicating that the model preserved a similar ability to rank late-delivery cases.

The test set was not used during feature selection, hyperparameter tuning, ensemble weighting, or threshold selection. Therefore, the reported test metrics represent the final evaluation of the selected model.
